In [ ]:
# Cell 1: Import semua library yang dibutuhkan

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.colors as mcolors
from PIL import Image
import os
import sys
import time
import struct
import zlib
from scipy.fftpack import dct, idct
from skimage.metrics import structural_similarity as ssim
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

In [ ]:
# Cell 2: Membuat citra uji dengan karakteristik berbeda

def create_test_images():
    images = {}
    size = 256

    gradient = np.zeros((size, size), dtype=np.uint8)
    for i in range(size):
        gradient[i, :] = int(i * 255 / size)
    images['gradient'] = gradient

    uniform_blocks = np.zeros((size, size), dtype=np.uint8)
    uniform_blocks[0:128, 0:128]     = 50
    uniform_blocks[0:128, 128:256]   = 150
    uniform_blocks[128:256, 0:128]   = 200
    uniform_blocks[128:256, 128:256] = 100
    images['uniform_blocks'] = uniform_blocks

    checkerboard = np.zeros((size, size), dtype=np.uint8)
    block_size = 32
    for i in range(0, size, block_size):
        for j in range(0, size, block_size):
            if (i // block_size + j // block_size) % 2 == 0:
                checkerboard[i:i+block_size, j:j+block_size] = 255
    images['checkerboard'] = checkerboard

    np.random.seed(42)
    images['random_noise'] = np.random.randint(0, 256, (size, size), dtype=np.uint8)

    geometric = np.ones((size, size), dtype=np.uint8) * 200
    cv2.circle(geometric, (128, 128), 80, 50, -1)
    cv2.rectangle(geometric, (50, 50), (120, 120), 150, -1)
    cv2.line(geometric, (0, 0), (255, 255), 30, 5)
    images['geometric'] = geometric

    return images

test_images = create_test_images()

# -------------------------------------------------------
# Visualisasi 1: Galeri citra uji
# -------------------------------------------------------
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
titles = ['Gradient', 'Uniform Blocks', 'Checkerboard', 'Random Noise', 'Geometric']
keys   = ['gradient', 'uniform_blocks', 'checkerboard', 'random_noise', 'geometric']
desc   = [
    'Baris seragam\n(ideal RLE)',
    'Blok warna\n(sangat ideal RLE)',
    'Pola berulang\n(cukup ideal RLE)',
    'Noise acak\n(sulit dikompres)',
    'Bentuk geometris\n(campuran)'
]

for col, (title, key, d) in enumerate(zip(titles, keys, desc)):
    img = test_images[key]

    # Baris atas: citra
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f'{title}\n{img.shape}', fontweight='bold')
    axes[0, col].axis('off')

    # Baris bawah: histogram
    axes[1, col].hist(img.flatten(), bins=64, color='steelblue',
                      edgecolor='navy', alpha=0.75, linewidth=0.4)
    axes[1, col].set_xlabel('Nilai Piksel')
    axes[1, col].set_ylabel('Frekuensi')
    axes[1, col].set_title(d, fontsize=9)
    axes[1, col].set_xlim(0, 255)
    axes[1, col].grid(True, alpha=0.3)

    unique_vals = len(np.unique(img))
    axes[1, col].text(0.98, 0.96, f'{unique_vals} nilai unik',
                      transform=axes[1, col].transAxes,
                      ha='right', va='top', fontsize=8,
                      color='darkred', fontweight='bold')

plt.suptitle('Galeri Citra Uji dan Distribusi Nilai Piksel',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cell2_test_images.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Profil piksel setiap citra
# -------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
row_mid = 128

for col, (title, key) in enumerate(zip(titles, keys)):
    img = test_images[key]
    axes[col].plot(img[row_mid, :], linewidth=1.2, color='#1565C0')
    axes[col].fill_between(range(256), img[row_mid, :], alpha=0.15, color='#1565C0')
    axes[col].set_title(f'{title}\n(profil baris tengah)', fontsize=9)
    axes[col].set_xlabel('Kolom piksel')
    axes[col].set_ylabel('Nilai piksel')
    axes[col].set_xlim(0, 255)
    axes[col].set_ylim(-5, 260)
    axes[col].grid(True, alpha=0.3)

plt.suptitle('Profil Nilai Piksel pada Baris Tengah (Baris ke-128)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell2_pixel_profiles.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Citra uji berhasil dibuat: {list(test_images.keys())}")

In [ ]:
# Cell 3: Implementasi dan visualisasi fungsi metrik evaluasi

def calculate_mse(original, reconstructed):
    original      = original.astype(np.float64)
    reconstructed = reconstructed.astype(np.float64)
    return np.mean((original - reconstructed) ** 2)

def calculate_psnr(original, reconstructed, max_pixel=255.0):
    mse = calculate_mse(original, reconstructed)
    if mse == 0:
        return float('inf')
    return 10 * np.log10((max_pixel ** 2) / mse)

def calculate_ssim(original, reconstructed):
    return ssim(original, reconstructed, data_range=255)

def calculate_compression_ratio(original_bytes, compressed_bytes):
    if compressed_bytes == 0:
        return float('inf')
    return original_bytes / compressed_bytes

def calculate_bit_rate(compressed_bytes, num_pixels):
    return (compressed_bytes * 8) / num_pixels

def get_image_size_bytes(image):
    return image.nbytes

def print_metrics(original, reconstructed,
                  original_bytes, compressed_bytes,
                  method_name="Kompresi"):
    mse_val   = calculate_mse(original, reconstructed)
    psnr_val  = calculate_psnr(original, reconstructed)
    ssim_val  = calculate_ssim(original, reconstructed)
    cr_val    = calculate_compression_ratio(original_bytes, compressed_bytes)
    bpp_val   = calculate_bit_rate(compressed_bytes, original.size)
    space_saved = (1 - compressed_bytes / original_bytes) * 100

    print(f"\n{'=' * 45}")
    print(f" Metrik Evaluasi: {method_name}")
    print(f"{'=' * 45}")
    print(f" Ukuran Asli        : {original_bytes:,} bytes")
    print(f" Ukuran Terkompresi : {compressed_bytes:,} bytes")
    print(f" Ruang Tersimpan    : {space_saved:.1f}%")
    print(f"{'=' * 45}")
    print(f" Compression Ratio  : {cr_val:.3f} : 1")
    print(f" Bit Per Pixel      : {bpp_val:.4f} bpp")
    print(f"{'=' * 45}")
    print(f" MSE                : {mse_val:.4f}")
    print(f" PSNR               : {psnr_val:.2f} dB")
    print(f" SSIM               : {ssim_val:.4f}")
    print(f"{'=' * 45}")

    return {
        'mse': mse_val, 'psnr': psnr_val, 'ssim': ssim_val,
        'compression_ratio': cr_val, 'bpp': bpp_val,
        'space_saved_percent': space_saved
    }

# -------------------------------------------------------
# Visualisasi 1: Ilustrasi konsep setiap metrik
# -------------------------------------------------------
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

ref_img = test_images['gradient'].copy()
np.random.seed(0)

noise_levels  = [0, 5, 20, 60]
noise_labels  = ['Asli (0)', 'Noise 5', 'Noise 20', 'Noise 60']
psnr_vals_demo = []
ssim_vals_demo = []
mse_vals_demo  = []

axes_top = [fig.add_subplot(gs[0, c]) for c in range(4)]
axes_bot = [fig.add_subplot(gs[1, c]) for c in range(4)]

for col, (noise, label) in enumerate(zip(noise_levels, noise_labels)):
    noisy = np.clip(
        ref_img.astype(np.int32) + np.random.randint(-noise, noise+1, ref_img.shape),
        0, 255
    ).astype(np.uint8)

    p = calculate_psnr(ref_img, noisy)
    s = calculate_ssim(ref_img, noisy)
    m = calculate_mse(ref_img, noisy)
    psnr_vals_demo.append(p if p != float('inf') else 60)
    ssim_vals_demo.append(s)
    mse_vals_demo.append(m)

    # Baris atas: citra
    axes_top[col].imshow(noisy, cmap='gray', vmin=0, vmax=255)
    psnr_str = f"{p:.1f}" if p != float('inf') else "inf"
    axes_top[col].set_title(
        f'{label}\nPSNR: {psnr_str} dB | SSIM: {s:.4f}',
        fontsize=9
    )
    axes_top[col].axis('off')

    # Baris bawah: error map
    err = np.abs(ref_img.astype(np.int32) - noisy.astype(np.int32)).astype(np.uint8)
    im  = axes_bot[col].imshow(err, cmap='hot', vmin=0, vmax=80)
    axes_bot[col].set_title(f'Error Map\nMSE: {m:.2f}', fontsize=9)
    axes_bot[col].axis('off')
    plt.colorbar(im, ax=axes_bot[col], shrink=0.75, label='|error|')

plt.suptitle('Ilustrasi Metrik Evaluasi: Pengaruh Tingkat Noise terhadap PSNR, SSIM, MSE',
             fontsize=13, fontweight='bold')
plt.savefig('cell3_metrics_illustration.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Grafik metrik vs tingkat noise
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x_noise = noise_levels

axes[0].plot(x_noise, mse_vals_demo, 'r-o', linewidth=2, markersize=8)
axes[0].fill_between(x_noise, mse_vals_demo, alpha=0.2, color='red')
axes[0].set_xlabel('Tingkat Noise')
axes[0].set_ylabel('MSE')
axes[0].set_title('MSE vs Tingkat Noise\n(Semakin besar = semakin buruk)')
axes[0].grid(True, alpha=0.3)
for x, y in zip(x_noise, mse_vals_demo):
    axes[0].annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)

axes[1].plot(x_noise, psnr_vals_demo, 'b-s', linewidth=2, markersize=8)
axes[1].fill_between(x_noise, psnr_vals_demo, alpha=0.2, color='blue')
axes[1].axhline(y=30, color='orange', linestyle='--', linewidth=1.5, label='30 dB (Baik)')
axes[1].axhline(y=40, color='green',  linestyle='--', linewidth=1.5, label='40 dB (Sangat Baik)')
axes[1].set_xlabel('Tingkat Noise')
axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('PSNR vs Tingkat Noise\n(Semakin besar = semakin baik)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
for x, y in zip(x_noise, psnr_vals_demo):
    axes[1].annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)

axes[2].plot(x_noise, ssim_vals_demo, 'g-^', linewidth=2, markersize=8)
axes[2].fill_between(x_noise, ssim_vals_demo, alpha=0.2, color='green')
axes[2].axhline(y=0.9, color='orange', linestyle='--', linewidth=1.5, label='SSIM 0.9')
axes[2].set_xlabel('Tingkat Noise')
axes[2].set_ylabel('SSIM')
axes[2].set_title('SSIM vs Tingkat Noise\n(1.0 = identik)')
axes[2].set_ylim(0, 1.1)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)
for x, y in zip(x_noise, ssim_vals_demo):
    axes[2].annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)

plt.suptitle('Perilaku Metrik Evaluasi terhadap Peningkatan Noise',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell3_metrics_graphs.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 3: Panduan interpretasi PSNR (gauge chart)
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
ax.set_facecolor('#F5F5F5')
fig.patch.set_facecolor('#F5F5F5')

ranges = [
    (0,  20, '#EF5350', 'Buruk\n< 20 dB'),
    (20, 30, '#FF9800', 'Cukup\n20-30 dB'),
    (30, 40, '#8BC34A', 'Baik\n30-40 dB'),
    (40, 55, '#2196F3', 'Sangat Baik\n> 40 dB'),
]
total = 55

for (lo, hi, color, label) in ranges:
    width = (hi - lo) / total
    x_start = lo / total
    rect = mpatches.FancyBboxPatch(
        (x_start + 0.005, 0.3), width - 0.01, 0.4,
        boxstyle="round,pad=0.01",
        transform=ax.transAxes,
        facecolor=color, edgecolor='white', linewidth=2,
        clip_on=False
    )
    ax.add_patch(rect)
    ax.text(x_start + width / 2, 0.50, label,
            transform=ax.transAxes,
            ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')
    ax.text(x_start, 0.22, f'{lo}', transform=ax.transAxes,
            ha='center', fontsize=9, color='#333333')

ax.text(1.0, 0.22, '55+', transform=ax.transAxes,
        ha='center', fontsize=9, color='#333333')
ax.text(0.5, 0.88, 'Panduan Interpretasi Nilai PSNR (dB)',
        transform=ax.transAxes, ha='center', fontsize=13,
        fontweight='bold', color='#212121')

plt.tight_layout()
plt.savefig('cell3_psnr_guide.png', dpi=100, bbox_inches='tight')
plt.show()

print("Fungsi metrik evaluasi berhasil dibuat dan divisualisasikan.")

In [ ]:
# Cell 4: Implementasi RLE Encoding dengan visualisasi langkah per langkah

def rle_encode_1d(data):
    if len(data) == 0:
        return []
    encoded = []
    count = 1
    current_val = data[0]
    for i in range(1, len(data)):
        if data[i] == current_val:
            count += 1
        else:
            encoded.append((int(current_val), count))
            current_val = data[i]
            count = 1
    encoded.append((int(current_val), count))
    return encoded

def rle_decode_1d(encoded_data, original_length):
    decoded = []
    for value, count in encoded_data:
        decoded.extend([value] * count)
    return np.array(decoded[:original_length], dtype=np.uint8)

def rle_encode_image(image):
    shape     = image.shape
    flat_data = image.flatten()
    encoded   = rle_encode_1d(flat_data)
    return encoded, shape, flat_data

def rle_decode_image(encoded_data, shape):
    total_pixels = shape[0] * shape[1]
    flat_decoded = rle_decode_1d(encoded_data, total_pixels)
    return flat_decoded.reshape(shape)

def calculate_rle_size(encoded_data):
    return len(encoded_data) * 3

# -------------------------------------------------------
# Visualisasi 1: Proses RLE langkah per langkah pada data 1D
# -------------------------------------------------------
sample_1d    = np.array([5,5,5,5, 3,3, 8,8,8,8,8, 2,2,2, 9, 7,7], dtype=np.uint8)
encoded_1d   = rle_encode_1d(sample_1d)
decoded_1d   = rle_decode_1d(encoded_1d, len(sample_1d))

fig, axes = plt.subplots(3, 1, figsize=(16, 9))
fig.patch.set_facecolor('#FAFAFA')

# Panel 1: Data asli dengan warna per run
run_colors = plt.cm.Set3(np.linspace(0, 1, len(encoded_1d)))
pos = 0
for ax_idx, (val, cnt) in enumerate(encoded_1d):
    for k in range(cnt):
        axes[0].add_patch(mpatches.FancyBboxPatch(
            (pos + 0.05, 0.1), 0.88, 0.8,
            boxstyle="round,pad=0.05",
            facecolor=run_colors[ax_idx], edgecolor='#333', linewidth=1.5
        ))
        axes[0].text(pos + 0.5, 0.5, str(val),
                     ha='center', va='center', fontsize=13, fontweight='bold')
        pos += 1

axes[0].set_xlim(0, len(sample_1d))
axes[0].set_ylim(0, 1)
axes[0].axis('off')
axes[0].set_title(f'DATA ASLI  ({len(sample_1d)} elemen) -- Warna berbeda = Run berbeda',
                  fontweight='bold', fontsize=11, pad=8)

# Panel 2: Hasil encoding (pasangan nilai, count)
for ax_idx, (val, cnt) in enumerate(encoded_1d):
    x = ax_idx * 1.5
    axes[1].add_patch(mpatches.FancyBboxPatch(
        (x + 0.05, 0.55), 1.4, 0.38,
        boxstyle="round,pad=0.04",
        facecolor=run_colors[ax_idx], edgecolor='#333', linewidth=1.5
    ))
    axes[1].text(x + 0.75, 0.74, f'val={val}',
                 ha='center', va='center', fontsize=10, fontweight='bold')

    axes[1].add_patch(mpatches.FancyBboxPatch(
        (x + 0.05, 0.10), 1.4, 0.38,
        boxstyle="round,pad=0.04",
        facecolor='white', edgecolor='#333', linewidth=1.5
    ))
    axes[1].text(x + 0.75, 0.29, f'count={cnt}',
                 ha='center', va='center', fontsize=10)

axes[1].set_xlim(0, len(encoded_1d) * 1.5)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title(
    f'HASIL RLE ENCODING  ({len(encoded_1d)} pasangan (nilai, count)) '
    f'-- Penghematan: {len(sample_1d)} -> {len(encoded_1d)*3} bytes',
    fontweight='bold', fontsize=11, pad=8
)

# Panel 3: Rekonstruksi (identik dengan asli)
pos = 0
for ax_idx, (val, cnt) in enumerate(encoded_1d):
    for k in range(cnt):
        axes[2].add_patch(mpatches.FancyBboxPatch(
            (pos + 0.05, 0.1), 0.88, 0.8,
            boxstyle="round,pad=0.05",
            facecolor=run_colors[ax_idx], edgecolor='#333', linewidth=1.5
        ))
        axes[2].text(pos + 0.5, 0.5, str(decoded_1d[pos]),
                     ha='center', va='center', fontsize=13, fontweight='bold')
        pos += 1

axes[2].set_xlim(0, len(sample_1d))
axes[2].set_ylim(0, 1)
axes[2].axis('off')
identical_str = 'IDENTIK' if np.array_equal(sample_1d, decoded_1d) else 'BERBEDA'
axes[2].set_title(
    f'HASIL DECODE -- {identical_str} dengan asli (LOSSLESS)',
    fontweight='bold', fontsize=11, pad=8,
    color='#2E7D32' if identical_str == 'IDENTIK' else '#C62828'
)

plt.suptitle('Proses Run-Length Encoding (RLE): Langkah per Langkah',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cell4_rle_step_by_step.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: RLE pada baris citra dan scan order
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

img_demo = test_images['uniform_blocks']

# Panel kiri: citra asli + arah scan
axes[0].imshow(img_demo, cmap='gray', vmin=0, vmax=255)
axes[0].annotate('', xy=(255, 0), xytext=(0, 0),
                  arrowprops=dict(arrowstyle='->', color='red', lw=3))
axes[0].annotate('', xy=(0, 1), xytext=(255, 1),
                  arrowprops=dict(arrowstyle='->', color='red', lw=2))
axes[0].set_title('Citra Asli + Arah Flatten (row-major)', fontweight='bold')
axes[0].set_xlabel('Kolom')
axes[0].set_ylabel('Baris')

# Panel tengah: data 1D (sampel)
flat = img_demo.flatten()
sample_flat = flat[:300]
encoded_flat = rle_encode_1d(sample_flat)
colors_flat  = plt.cm.tab10(np.arange(len(encoded_flat)) % 10)

pos = 0
for i, (val, cnt) in enumerate(encoded_flat):
    axes[1].axvspan(pos, pos + cnt, alpha=0.5,
                    color=colors_flat[i], label=f'({val},{cnt})')
    axes[1].text(pos + cnt / 2, val + 5, f'{cnt}',
                 ha='center', fontsize=8, color='black')
    pos += cnt

axes[1].plot(range(300), sample_flat[:300], 'k-', linewidth=1, zorder=5)
axes[1].set_xlabel('Indeks piksel (300 pertama)')
axes[1].set_ylabel('Nilai piksel')
axes[1].set_title('Visualisasi Run pada Data 1D\n(Warna = Run berbeda)', fontweight='bold')
axes[1].set_xlim(0, 300)
axes[1].set_ylim(-10, 270)
axes[1].grid(True, alpha=0.3)

# Panel kanan: pie chart komposisi run
run_lengths_all = [c for _, c in rle_encode_image(img_demo)[0]]
run_lengths_arr = np.array(run_lengths_all)

bins   = [1, 2, 5, 10, 50, max(run_lengths_arr)+1]
labels = ['1', '2-4', '5-9', '10-49', f'50+']
counts = [np.sum((run_lengths_arr >= bins[i]) & (run_lengths_arr < bins[i+1]))
          for i in range(len(bins)-1)]
pie_colors = ['#EF5350','#FF9800','#FFEB3B','#8BC34A','#2196F3']

wedges, texts, autotexts = axes[2].pie(
    counts, labels=labels, colors=pie_colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(9)
axes[2].set_title('Distribusi Panjang Run\n(Citra Uniform Blocks)', fontweight='bold')

plt.suptitle('Mekanisme RLE pada Citra 2D', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell4_rle_mechanism.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 3: Ringkasan hasil encode semua citra
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
img_names_list = list(test_images.keys())
cr_list, run_list, hemat_list = [], [], []

for name in img_names_list:
    img = test_images[name]
    enc, shape, _ = rle_encode_image(img)
    orig_b = img.nbytes
    comp_b = calculate_rle_size(enc)
    cr_list.append(calculate_compression_ratio(orig_b, comp_b))
    run_list.append(len(enc))
    hemat_list.append((1 - comp_b/orig_b)*100)

bar_colors = ['#2196F3' if cr > 1 else '#EF5350' for cr in cr_list]

axes[0].barh(img_names_list, cr_list, color=bar_colors,
             edgecolor='black', linewidth=0.8)
axes[0].axvline(x=1, color='red', linestyle='--', linewidth=2, label='CR = 1')
axes[0].set_xlabel('Compression Ratio')
axes[0].set_title('Compression Ratio per Citra\n(Biru=efektif, Merah=tidak)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(cr_list):
    axes[0].text(v + 0.02, i, f'{v:.2f}x', va='center', fontsize=9)

axes[1].barh(img_names_list, run_list, color='#FF9800',
             edgecolor='black', linewidth=0.8)
axes[1].set_xlabel('Jumlah Run (pasangan)')
axes[1].set_title('Jumlah Run setelah RLE\n(Lebih sedikit = lebih padat)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(run_list):
    axes[1].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

color_hemat = ['#4CAF50' if h > 0 else '#EF5350' for h in hemat_list]
axes[2].barh(img_names_list, hemat_list, color=color_hemat,
             edgecolor='black', linewidth=0.8)
axes[2].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[2].set_xlabel('Penghematan Ruang (%)')
axes[2].set_title('Penghematan Ruang\n(Positif = menguntungkan)', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(hemat_list):
    xpos = v + 0.5 if v >= 0 else v - 4
    axes[2].text(xpos, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Ringkasan Hasil RLE Encoding pada Semua Citra Uji',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell4_rle_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print("Implementasi RLE selesai. Semua visualisasi ditampilkan.")

In [ ]:
# Cell 5: Visualisasi proses RLE secara detail per citra

def visualize_rle_process(image, image_name="Citra"):
    encoded, shape, flat_data = rle_encode_image(image)
    reconstructed = rle_decode_image(encoded, shape)
    run_lengths   = [count for _, count in encoded]
    run_values    = [val   for val,  _ in encoded]

    original_bytes   = image.nbytes
    compressed_bytes = calculate_rle_size(encoded)
    cr  = calculate_compression_ratio(original_bytes, compressed_bytes)
    mse_val  = calculate_mse(image, reconstructed)
    psnr_val = calculate_psnr(image, reconstructed)
    ssim_val = calculate_ssim(image, reconstructed)

    fig = plt.figure(figsize=(20, 14))
    gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # (0,0) Citra asli
    ax00 = fig.add_subplot(gs[0, 0])
    ax00.imshow(image, cmap='gray', vmin=0, vmax=255)
    ax00.set_title(f'Citra Asli\n{image.shape} | {original_bytes:,} bytes',
                   fontweight='bold')
    ax00.axis('off')

    # (0,1) Citra rekonstruksi
    ax01 = fig.add_subplot(gs[0, 1])
    ax01.imshow(reconstructed, cmap='gray', vmin=0, vmax=255)
    ax01.set_title('Citra Rekonstruksi\n(LOSSLESS - Identik)', fontweight='bold',
                   color='#2E7D32')
    ax01.axis('off')

    # (0,2) Error map
    ax02 = fig.add_subplot(gs[0, 2])
    error_map = np.abs(image.astype(np.int32) - reconstructed.astype(np.int32))
    im02 = ax02.imshow(error_map, cmap='hot', vmin=0, vmax=5)
    ax02.set_title(f'Error Map\nMSE={mse_val:.6f} (seharusnya 0)', fontweight='bold')
    ax02.axis('off')
    plt.colorbar(im02, ax=ax02, shrink=0.75)

    # (0,3) Info panel
    ax03 = fig.add_subplot(gs[0, 3])
    ax03.axis('off')
    ax03.set_facecolor('#F1F8E9')
    space_saved = (1 - compressed_bytes / original_bytes) * 100
    info = (
        f"STATISTIK RLE\n"
        f"{'='*22}\n"
        f"Asli: {original_bytes:,} bytes\n"
        f"RLE:  {compressed_bytes:,} bytes\n\n"
        f"Total Run: {len(encoded):,}\n"
        f"CR: {cr:.3f} : 1\n"
        f"Hemat: {space_saved:.1f}%\n\n"
        f"PSNR: {'inf' if psnr_val==float('inf') else f'{psnr_val:.2f}'} dB\n"
        f"SSIM: {ssim_val:.6f}\n"
        f"MSE:  {mse_val:.6f}\n\n"
        f"Lossless: YA"
    )
    ax03.text(0.08, 0.96, info, transform=ax03.transAxes,
              fontsize=9.5, va='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.9))

    # (1,0:2) Perbandingan profil piksel
    ax10 = fig.add_subplot(gs[1, :2])
    row_mid = image.shape[0] // 2
    ax10.plot(image[row_mid, :], 'b-', linewidth=2, label='Asli', alpha=0.8)
    ax10.plot(reconstructed[row_mid, :], 'r--', linewidth=2,
              label='Rekonstruksi', alpha=0.7)
    ax10.set_xlabel('Kolom piksel')
    ax10.set_ylabel('Nilai piksel')
    ax10.set_title(f'Perbandingan Profil Piksel (Baris ke-{row_mid})\n'
                   f'Kedua garis seharusnya berimpit sempurna', fontweight='bold')
    ax10.legend()
    ax10.grid(True, alpha=0.3)
    ax10.set_xlim(0, image.shape[1])
    ax10.set_ylim(-5, 265)

    # (1,2:4) Panjang run per indeks
    ax11 = fig.add_subplot(gs[1, 2:])
    max_show = min(150, len(run_lengths))
    x_run    = np.arange(max_show)
    run_cmap = plt.cm.plasma(np.array(run_lengths[:max_show]) /
                              max(run_lengths[:max_show]))
    ax11.bar(x_run, run_lengths[:max_show], color=run_cmap,
             edgecolor='none', width=1.0)
    ax11.set_xlabel(f'Indeks Run (150 pertama dari {len(run_lengths):,} total)')
    ax11.set_ylabel('Panjang Run')
    ax11.set_title('Panjang Setiap Run\n(Tinggi bar = berapa piksel yang direpresentasikan)',
                   fontweight='bold')
    ax11.grid(True, alpha=0.3, axis='y')

    sm = plt.cm.ScalarMappable(cmap='plasma',
                                norm=mcolors.Normalize(vmin=0,
                                                       vmax=max(run_lengths[:max_show])))
    plt.colorbar(sm, ax=ax11, label='Panjang Run', shrink=0.75)

    # (2,0:2) Histogram panjang run + statistik
    ax20 = fig.add_subplot(gs[2, :2])
    ax20.hist(run_lengths, bins=40, color='coral', edgecolor='darkred',
              alpha=0.75, linewidth=0.5, density=False)
    ax20.axvline(np.mean(run_lengths), color='blue', linestyle='--',
                 linewidth=2.5, label=f'Mean: {np.mean(run_lengths):.1f}')
    ax20.axvline(np.median(run_lengths), color='green', linestyle='-.',
                 linewidth=2.5, label=f'Median: {np.median(run_lengths):.0f}')
    ax20.axvline(max(run_lengths), color='red', linestyle=':',
                 linewidth=2.5, label=f'Max: {max(run_lengths)}')
    ax20.set_xlabel('Panjang Run')
    ax20.set_ylabel('Jumlah Run')
    ax20.set_title('Histogram Distribusi Panjang Run\n'
                   '(Run panjang = area seragam = RLE efektif)', fontweight='bold')
    ax20.legend()
    ax20.grid(True, alpha=0.3)

    # (2,2:4) Scatter: nilai run vs panjang run
    ax21 = fig.add_subplot(gs[2, 2:])
    sc = ax21.scatter(run_values[:500], run_lengths[:500],
                      c=run_lengths[:500], cmap='YlOrRd',
                      alpha=0.6, s=20, edgecolors='none')
    ax21.set_xlabel('Nilai Piksel Run')
    ax21.set_ylabel('Panjang Run')
    ax21.set_title('Nilai Piksel vs Panjang Run (500 Run Pertama)\n'
                   'Titik tinggi = nilai piksel dengan area seragam panjang', fontweight='bold')
    ax21.grid(True, alpha=0.3)
    plt.colorbar(sc, ax=ax21, label='Panjang Run', shrink=0.75)

    plt.suptitle(f'Analisis Mendalam RLE: {image_name}',
                 fontsize=14, fontweight='bold')
    plt.savefig(f'cell5_rle_detail_{image_name.lower().replace(" ", "_")}.png',
                dpi=100, bbox_inches='tight')
    plt.show()

    print(f"[{image_name}] CR={cr:.2f} | Runs={len(encoded):,} | "
          f"Mean run={np.mean(run_lengths):.1f} | Max run={max(run_lengths)}")

print("Visualisasi RLE untuk Citra UNIFORM BLOCKS (kasus terbaik):")
visualize_rle_process(test_images['uniform_blocks'], 'Uniform Blocks')

print("\nVisualisasi RLE untuk Citra RANDOM NOISE (kasus terburuk):")
visualize_rle_process(test_images['random_noise'], 'Random Noise')

print("\nVisualisasi RLE untuk Citra GRADIENT:")
visualize_rle_process(test_images['gradient'], 'Gradient')

In [ ]:
# Cell 6: Perbandingan efektivitas RLE + visualisasi lengkap

def compare_rle_effectiveness(images_dict):
    results = {}
    print("PERBANDINGAN EFEKTIVITAS RLE")
    print("=" * 65)
    print(f"{'Citra':<18} {'Asli(B)':>9} {'RLE(B)':>9} "
          f"{'CR':>7} {'Hemat':>8} {'Runs':>8}")
    print("-" * 65)

    for name, img in images_dict.items():
        encoded, shape, _ = rle_encode_image(img)
        reconstructed     = rle_decode_image(encoded, shape)
        orig_b = img.nbytes
        comp_b = calculate_rle_size(encoded)
        cr     = calculate_compression_ratio(orig_b, comp_b)
        hemat  = (1 - comp_b / orig_b) * 100
        runs   = len(encoded)

        print(f"{name:<18} {orig_b:>9,} {comp_b:>9,} "
              f"{cr:>7.2f} {hemat:>7.1f}% {runs:>8,}")

        results[name] = {
            'original_bytes': orig_b, 'compressed_bytes': comp_b,
            'compression_ratio': cr,  'space_saved': hemat,
            'num_runs': runs,
            'mean_run_length': np.mean([c for _, c in encoded])
        }
    print("-" * 65)
    return results

comparison_results = compare_rle_effectiveness(test_images)

# -------------------------------------------------------
# Visualisasi 1: Dashboard perbandingan RLE
# -------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#FAFAFA')

names    = list(comparison_results.keys())
cr_vals  = [comparison_results[n]['compression_ratio'] for n in names]
hemat    = [comparison_results[n]['space_saved']        for n in names]
runs_val = [comparison_results[n]['num_runs']            for n in names]
mean_run = [comparison_results[n]['mean_run_length']     for n in names]
orig_b   = [comparison_results[n]['original_bytes']      for n in names]
comp_b   = [comparison_results[n]['compressed_bytes']    for n in names]

c_cr  = ['#2196F3' if c > 1 else '#EF5350' for c in cr_vals]
c_hem = ['#4CAF50' if h > 0 else '#EF5350' for h in hemat]

# Plot CR
bars = axes[0, 0].bar(names, cr_vals, color=c_cr, edgecolor='black', linewidth=0.8)
axes[0, 0].axhline(y=1, color='red', linestyle='--', linewidth=2.5, label='CR = 1')
axes[0, 0].set_title('Compression Ratio\n(> 1 = menguntungkan)', fontweight='bold')
axes[0, 0].set_ylabel('Compression Ratio')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, cr_vals):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                    f'{val:.2f}x', ha='center', fontsize=9, fontweight='bold')

# Plot Penghematan
bars2 = axes[0, 1].bar(names, hemat, color=c_hem, edgecolor='black', linewidth=0.8)
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[0, 1].set_title('Penghematan Ruang (%)\n(Positif = ukuran berkurang)', fontweight='bold')
axes[0, 1].set_ylabel('Penghematan (%)')
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, hemat):
    ypos = bar.get_height() + 0.5 if val >= 0 else bar.get_height() - 3.5
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, ypos,
                    f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Plot Jumlah Run
bars3 = axes[0, 2].bar(names, runs_val, color='#FF9800', edgecolor='black', linewidth=0.8)
axes[0, 2].set_title('Jumlah Run (Pasangan)\n(Lebih sedikit = lebih efisien)', fontweight='bold')
axes[0, 2].set_ylabel('Jumlah Run')
axes[0, 2].grid(True, alpha=0.3, axis='y')
axes[0, 2].tick_params(axis='x', rotation=20)
for bar, val in zip(bars3, runs_val):
    axes[0, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                    f'{val:,}', ha='center', fontsize=8.5)

# Plot Mean Run Length
bars4 = axes[1, 0].bar(names, mean_run, color='#9C27B0', edgecolor='black', linewidth=0.8)
axes[1, 0].set_title('Rata-rata Panjang Run\n(Lebih panjang = lebih baik untuk RLE)',
                      fontweight='bold')
axes[1, 0].set_ylabel('Rata-rata Panjang Run')
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars4, mean_run):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')

# Plot ukuran sebelum vs sesudah (grouped bar)
x_idx   = np.arange(len(names))
w_bar   = 0.35
bars_o  = axes[1, 1].bar(x_idx - w_bar/2, orig_b, w_bar,
                          label='Asli', color='#1565C0', edgecolor='black', linewidth=0.8)
bars_c  = axes[1, 1].bar(x_idx + w_bar/2, comp_b, w_bar,
                          label='RLE', color='#EF5350', edgecolor='black', linewidth=0.8)
axes[1, 1].set_xticks(x_idx)
axes[1, 1].set_xticklabels(names, rotation=20, ha='right')
axes[1, 1].set_title('Perbandingan Ukuran\nAsli vs RLE (bytes)', fontweight='bold')
axes[1, 1].set_ylabel('Ukuran (bytes)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Plot scatter CR vs mean run length
sc = axes[1, 2].scatter(mean_run, cr_vals, c=range(len(names)),
                         cmap='tab10', s=200, edgecolors='black',
                         linewidth=1.5, zorder=5)
axes[1, 2].axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='CR = 1')
for i, name in enumerate(names):
    axes[1, 2].annotate(name, (mean_run[i], cr_vals[i]),
                         textcoords='offset points', xytext=(8, 5), fontsize=8)
axes[1, 2].set_xlabel('Rata-rata Panjang Run')
axes[1, 2].set_ylabel('Compression Ratio')
axes[1, 2].set_title('Mean Run Length vs CR\n(Korelasi positif diharapkan)', fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Dashboard Perbandingan Efektivitas RLE pada Berbagai Jenis Citra',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cell6_rle_comparison_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Heatmap perbandingan visual citra asli vs rekonstruksi
# -------------------------------------------------------
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
keys_list  = list(test_images.keys())
title_list = ['Gradient','Uniform Blocks','Checkerboard','Random Noise','Geometric']

for col, (key, title) in enumerate(zip(keys_list, title_list)):
    img  = test_images[key]
    enc, shp, _ = rle_encode_image(img)
    recon = rle_decode_image(enc, shp)
    diff  = np.abs(img.astype(np.int32) - recon.astype(np.int32))
    cr    = calculate_compression_ratio(img.nbytes, calculate_rle_size(enc))

    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f'{title}\nCR = {cr:.2f}x', fontweight='bold', fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(recon, cmap='gray', vmin=0, vmax=255)
    axes[1, col].set_title('Rekonstruksi\n(Lossless)', fontsize=9, color='#2E7D32')
    axes[1, col].axis('off')

    im_diff = axes[2, col].imshow(diff, cmap='hot', vmin=0, vmax=1)
    axes[2, col].set_title(f'Error Map\nMSE={calculate_mse(img,recon):.6f}', fontsize=9)
    axes[2, col].axis('off')
    plt.colorbar(im_diff, ax=axes[2, col], shrink=0.7)

for row, label in enumerate(['Citra Asli','Rekonstruksi RLE','Error Map (MSE=0)']):
    fig.text(0.01, 0.82 - row * 0.30, label, fontsize=11,
             fontweight='bold', rotation=90, va='center', color='#1565C0')

plt.suptitle('Verifikasi Lossless RLE: Asli vs Rekonstruksi vs Error\n'
             '(Error map seharusnya seluruhnya hitam - tidak ada perbedaan)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig('cell6_rle_lossless_verification.png', dpi=100, bbox_inches='tight')
plt.show()

print("Perbandingan RLE selesai. RLE bersifat lossless pada semua jenis citra.")

In [ ]:
# Cell 7: Implementasi DCT dengan visualisasi basis dan transformasi

JPEG_QUANTIZATION_MATRIX = np.array([
    [16,11,10,16,24, 40, 51, 61],
    [12,12,14,19,26, 58, 60, 55],
    [14,13,16,24,40, 57, 69, 56],
    [14,17,22,29,51, 87, 80, 62],
    [18,22,37,56,68,109,103, 77],
    [24,35,55,64,81,104,113, 92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103, 99]
], dtype=np.float64)

def apply_dct_2d(block):
    return dct(dct(block.T, norm='ortho').T, norm='ortho')

def apply_idct_2d(dct_block):
    return idct(idct(dct_block.T, norm='ortho').T, norm='ortho')

def get_quantization_matrix(quality_factor=50):
    quality_factor = max(1, min(100, quality_factor))
    scale  = 5000 / quality_factor if quality_factor < 50 else 200 - 2 * quality_factor
    q_mat  = np.floor((JPEG_QUANTIZATION_MATRIX * scale + 50) / 100)
    return np.clip(q_mat, 1, 255)

def quantize_block(dct_block, q_matrix):
    return np.round(dct_block / q_matrix).astype(np.int32)

def dequantize_block(quantized_block, q_matrix):
    return (quantized_block * q_matrix).astype(np.float64)

def dct_compress_image(image, block_size=8, quality_factor=50):
    h, w   = image.shape
    pad_h  = (block_size - h % block_size) % block_size
    pad_w  = (block_size - w % block_size) % block_size
    padded = np.pad(image, ((0,pad_h),(0,pad_w)), mode='edge')
    ph, pw = padded.shape
    img_float = padded.astype(np.float64) - 128.0
    q_matrix  = get_quantization_matrix(quality_factor)
    quantized_blocks = np.zeros_like(padded, dtype=np.int32)
    nonzero_count    = 0
    for i in range(0, ph, block_size):
        for j in range(0, pw, block_size):
            block   = img_float[i:i+block_size, j:j+block_size]
            dct_c   = apply_dct_2d(block)
            q_block = quantize_block(dct_c, q_matrix)
            quantized_blocks[i:i+block_size, j:j+block_size] = q_block
            nonzero_count += np.count_nonzero(q_block)
    return quantized_blocks, q_matrix, (ph, pw), nonzero_count

def dct_decompress_image(quantized_blocks, q_matrix, original_shape, block_size=8):
    ph, pw = quantized_blocks.shape
    recon  = np.zeros_like(quantized_blocks, dtype=np.float64)
    for i in range(0, ph, block_size):
        for j in range(0, pw, block_size):
            q_block = quantized_blocks[i:i+block_size, j:j+block_size]
            dct_c   = dequantize_block(q_block, q_matrix)
            block   = apply_idct_2d(dct_c)
            recon[i:i+block_size, j:j+block_size] = block
    recon = np.clip(recon + 128.0, 0, 255).astype(np.uint8)
    oh, ow = original_shape
    return recon[:oh, :ow]

def estimate_dct_compressed_size(nonzero_count, total_blocks):
    return nonzero_count * 2 + 500

# -------------------------------------------------------
# Visualisasi 1: Basis DCT 2D (8x8)
# -------------------------------------------------------
fig, axes = plt.subplots(8, 8, figsize=(14, 14))
fig.patch.set_facecolor('#111111')

for u in range(8):
    for v in range(8):
        basis = np.zeros((8, 8))
        basis[u, v] = 1.0
        basis_img  = apply_idct_2d(basis)
        axes[u, v].imshow(basis_img, cmap='gray', vmin=-0.5, vmax=0.5)
        axes[u, v].axis('off')
        if u == 0:
            axes[u, v].set_title(f'v={v}', color='white', fontsize=7, pad=2)
        if v == 0:
            axes[u, v].set_ylabel(f'u={u}', color='white', fontsize=7,
                                   rotation=0, labelpad=18)

fig.text(0.5, 0.98, '64 Basis Function DCT 2D (8x8)\n'
         '(Kiri atas = frekuensi rendah, Kanan bawah = frekuensi tinggi)',
         ha='center', va='top', fontsize=12, color='white', fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('cell7_dct_basis.png', dpi=100, bbox_inches='tight',
            facecolor='#111111')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: DCT satu blok lengkap step-by-step
# -------------------------------------------------------
img_demo  = test_images['gradient']
cy, cx    = img_demo.shape[0]//2, img_demo.shape[1]//2
blk_orig  = img_demo[cy:cy+8, cx:cx+8].astype(np.float64)
blk_shift = blk_orig - 128.0
dct_coeff = apply_dct_2d(blk_shift)
q_mat_50  = get_quantization_matrix(50)
q_block   = quantize_block(dct_coeff, q_mat_50)
deq_block = dequantize_block(q_block, q_mat_50)
recon_blk = np.clip(apply_idct_2d(deq_block) + 128.0, 0, 255)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.patch.set_facecolor('#F8F9FA')

steps_data = [
    (blk_orig,             'gray',   0,   255, 'Langkah 1\nBlok Asli (8x8)'),
    (blk_shift,            'RdBu',  -128, 128, 'Langkah 2\nShift -128\n(center ke nol)'),
    (dct_coeff,            'RdBu',  -200, 200, 'Langkah 3\nKoefisien DCT\n(domain frekuensi)'),
    (q_block.astype(float),'RdBu',   -50,  50, 'Langkah 4\nSetelah Kuantisasi\n(Q=50, lossy)'),
    (deq_block,            'RdBu',  -200, 200, 'Langkah 5\nDe-Kuantisasi\n(domain frekuensi)'),
]
recon_steps_data = [
    (recon_blk,              'gray',   0, 255, 'Langkah 6\nRekonstruksi\n(setelah IDCT)'),
    (np.abs(blk_orig-recon_blk), 'hot', 0,  20, 'Langkah 7\nError per Piksel\n|Asli - Rekonstruksi|'),
    (q_mat_50,               'YlOrRd', 0, 120, 'Referensi\nMatriks Kuantisasi\n(QF=50)'),
    (np.log1p(np.abs(dct_coeff)), 'hot', 0, 6,'Koefisien DCT\n(log scale)\nFrekuensi tinggi kanan-bawah'),
    (np.abs(q_block).astype(float), 'YlOrRd', 0, 30,
     f'Koefisien Non-zero\n= {np.count_nonzero(q_block)}/64\n({np.count_nonzero(q_block)/64*100:.0f}%)'),
]

for col, (data, cmap, vmin, vmax, title) in enumerate(steps_data):
    im = axes[0, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                              interpolation='nearest')
    axes[0, col].set_title(title, fontsize=9, fontweight='bold')
    for i in range(8):
        for j in range(8):
            axes[0, col].text(j, i, f'{data[i,j]:.0f}',
                               ha='center', va='center', fontsize=6.5,
                               color='black' if abs(data[i,j]) < (vmax*0.6) else 'white')
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], shrink=0.65)

for col, (data, cmap, vmin, vmax, title) in enumerate(recon_steps_data):
    im = axes[1, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                              interpolation='nearest')
    axes[1, col].set_title(title, fontsize=9, fontweight='bold')
    for i in range(8):
        for j in range(8):
            axes[1, col].text(j, i, f'{data[i,j]:.0f}',
                               ha='center', va='center', fontsize=6.5,
                               color='black' if data[i,j] < vmax*0.6 else 'white')
    axes[1, col].axis('off')
    plt.colorbar(im, ax=axes[1, col], shrink=0.65)

plt.suptitle('Proses DCT Compression Lengkap pada Satu Blok 8x8\n'
             '(Baris atas: Encoding | Baris bawah: Decoding & Analisis)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell7_dct_block_process.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 3: Matriks kuantisasi berbagai QF
# -------------------------------------------------------
qf_vals = [10, 25, 50, 75, 90]
fig, axes = plt.subplots(1, len(qf_vals), figsize=(18, 4))

for ax, qf in zip(axes, qf_vals):
    qm = get_quantization_matrix(qf)
    im = ax.imshow(qm, cmap='YlOrRd', vmin=1, vmax=255)
    ax.set_title(f'Quality Factor = {qf}\n'
                 f'Min={qm.min():.0f}, Max={qm.max():.0f}',
                 fontweight='bold')
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f'{qm[i,j]:.0f}', ha='center', va='center',
                    fontsize=8, color='black' if qm[i,j] < 150 else 'white')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.75)

plt.suptitle('Matriks Kuantisasi pada Berbagai Quality Factor\n'
             '(Nilai besar = membuang lebih banyak informasi frekuensi tersebut)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell7_quantization_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

print("Implementasi DCT dan visualisasi selesai.")

In [ ]:
# Cell 8: Eksperimen DCT dengan berbagai quality factor + visualisasi

def experiment_dct_quality(image, quality_factors, image_name="Citra"):
    results      = {}
    original_bytes = image.nbytes
    h, w         = image.shape
    block_size   = 8
    total_blocks = (h // block_size + 1) * (w // block_size + 1)

    print(f"\nEksperimen DCT: {image_name} | Asli: {original_bytes:,} bytes")
    print("=" * 75)
    print(f"{'QF':>6} {'Bytes':>10} {'CR':>7} {'MSE':>10} {'PSNR':>8} {'SSIM':>7} {'NonZero%':>9}")
    print("-" * 75)

    for qf in quality_factors:
        q_blocks, q_mat, padded_shape, nonzero = dct_compress_image(
            image, block_size=block_size, quality_factor=qf
        )
        reconstructed    = dct_decompress_image(q_blocks, q_mat, image.shape, block_size)
        compressed_bytes = estimate_dct_compressed_size(nonzero, total_blocks)
        cr               = calculate_compression_ratio(original_bytes, compressed_bytes)
        mse_val          = calculate_mse(image, reconstructed)
        psnr_val         = calculate_psnr(image, reconstructed)
        ssim_val         = calculate_ssim(image, reconstructed)
        nz_pct           = nonzero / (image.size) * 100

        results[qf] = {
            'reconstructed': reconstructed,
            'compressed_bytes': compressed_bytes,
            'compression_ratio': cr,
            'mse': mse_val, 'psnr': psnr_val, 'ssim': ssim_val,
            'nonzero_count': nonzero, 'nonzero_pct': nz_pct
        }
        print(f"{qf:>6} {compressed_bytes:>10,} {cr:>7.2f} "
              f"{mse_val:>10.2f} {psnr_val:>8.2f} {ssim_val:>7.4f} {nz_pct:>8.1f}%")

    print("=" * 75)
    return results

test_img     = test_images['gradient']
quality_list = [1, 5, 10, 20, 30, 50, 70, 80, 90, 95, 100]
dct_results  = experiment_dct_quality(test_img, quality_list, "Gradient")

# -------------------------------------------------------
# Visualisasi 1: Strip citra rekonstruksi dengan quality bar
# -------------------------------------------------------
quality_display = [1, 5, 10, 30, 50, 70, 90, 100]
n_disp = len(quality_display)

fig, axes = plt.subplots(3, n_disp, figsize=(22, 9))
fig.patch.set_facecolor('#F0F0F0')

# Baris citra asli (hanya kolom pertama)
for col, qf in enumerate(quality_display):
    res   = dct_results[qf]
    recon = res['reconstructed']

    # Tentukan warna judul berdasarkan kualitas PSNR
    psnr_val = res['psnr']
    if psnr_val >= 40:
        title_color = '#2E7D32'
    elif psnr_val >= 30:
        title_color = '#F57F17'
    elif psnr_val >= 20:
        title_color = '#E65100'
    else:
        title_color = '#B71C1C'

    # Baris 1: citra rekonstruksi
    axes[0, col].imshow(recon, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(
        f'QF = {qf}\nCR: {res["compression_ratio"]:.1f}x',
        fontweight='bold', fontsize=9, color=title_color
    )
    axes[0, col].axis('off')

    # Baris 2: error map
    err = np.abs(test_img.astype(np.int32) - recon.astype(np.int32))
    im  = axes[1, col].imshow(err, cmap='hot', vmin=0, vmax=40)
    axes[1, col].set_title(
        f'PSNR: {psnr_val:.1f} dB\nSSIM: {res["ssim"]:.3f}',
        fontsize=9, color=title_color
    )
    axes[1, col].axis('off')

    # Baris 3: histogram error
    axes[2, col].hist(err.flatten(), bins=40, color='coral',
                       edgecolor='darkred', linewidth=0.5, alpha=0.8)
    axes[2, col].set_title(f'Distribusi Error\nMSE={res["mse"]:.1f}',
                            fontsize=8)
    axes[2, col].set_xlabel('|error|', fontsize=8)
    axes[2, col].tick_params(labelsize=7)
    axes[2, col].grid(True, alpha=0.3)
    axes[2, col].set_yscale('log')

row_labels = ['Rekonstruksi', 'Error Map', 'Histogram Error']
for row, label in enumerate(row_labels):
    fig.text(0.005, 0.82 - row * 0.30, label, fontsize=10,
             fontweight='bold', rotation=90, va='center', color='#1565C0')

plt.suptitle('Kompresi DCT: Pengaruh Quality Factor terhadap Kualitas Visual\n'
             '(Hijau = PSNR>40 dB | Kuning = 30-40 | Oranye = 20-30 | Merah = <20)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0.02, 0, 1, 0.95])
plt.savefig('cell8_dct_quality_strips.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Spektrum frekuensi setelah kuantisasi per QF
# -------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
qf_display2 = [5, 10, 30, 50, 70, 90, 95, 100]

for idx, qf in enumerate(qf_display2):
    row = idx // 4
    col = idx % 4
    q_blocks, q_mat, _, _ = dct_compress_image(test_img, quality_factor=qf)

    # Hitung distribusi non-zero per posisi frekuensi
    h_blk = (test_img.shape[0] // 8) * 8
    w_blk = (test_img.shape[1] // 8) * 8
    nonzero_map = np.zeros((8, 8))
    n_blocks = 0
    for i in range(0, h_blk, 8):
        for j in range(0, w_blk, 8):
            blk = q_blocks[i:i+8, j:j+8]
            nonzero_map += (blk != 0).astype(float)
            n_blocks += 1
    nonzero_pct_map = nonzero_map / n_blocks * 100

    im = axes[row, col].imshow(nonzero_pct_map, cmap='hot', vmin=0, vmax=100)
    axes[row, col].set_title(f'QF = {qf}\n'
                              f'Total nonzero: {nonzero_map.sum():.0f} / {n_blocks*64}',
                              fontweight='bold', fontsize=9)
    for i in range(8):
        for j in range(8):
            axes[row, col].text(j, i, f'{nonzero_pct_map[i,j]:.0f}%',
                                 ha='center', va='center', fontsize=7,
                                 color='white' if nonzero_pct_map[i,j] > 50 else 'black')
    axes[row, col].axis('off')
    plt.colorbar(im, ax=axes[row, col], shrink=0.7, label='% nonzero')

plt.suptitle('Peta Non-zero Koefisien DCT per Posisi Frekuensi (%)\n'
             '(DC=kiri atas, frekuensi tinggi=kanan bawah | Putih=selalu ada, Hitam=selalu nol)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cell8_dct_frequency_map.png', dpi=100, bbox_inches='tight')
plt.show()

print("Eksperimen quality factor DCT selesai.")

In [ ]:
# Cell 9: Visualisasi trade-off kualitas vs kompresi

def plot_quality_tradeoff(dct_results, quality_factors):
    qfs       = sorted(dct_results.keys())
    psnr_vals = [dct_results[qf]['psnr']              for qf in qfs]
    ssim_vals = [dct_results[qf]['ssim']              for qf in qfs]
    cr_vals   = [dct_results[qf]['compression_ratio'] for qf in qfs]
    mse_vals  = [dct_results[qf]['mse']               for qf in qfs]
    nz_pct    = [dct_results[qf]['nonzero_pct']       for qf in qfs]

    fig = plt.figure(figsize=(20, 14))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

    # Plot 1: PSNR vs QF
    ax1 = fig.add_subplot(gs[0, 0])
    line1, = ax1.plot(qfs, psnr_vals, 'b-o', linewidth=2.5, markersize=8, zorder=5)
    ax1.fill_between(qfs, psnr_vals, alpha=0.15, color='blue')
    ax1.axhspan(40, max(psnr_vals)+2, alpha=0.07, color='green',  label='Sangat Baik (>40)')
    ax1.axhspan(30, 40,               alpha=0.07, color='yellow', label='Baik (30-40)')
    ax1.axhspan(20, 30,               alpha=0.07, color='orange', label='Cukup (20-30)')
    ax1.axhspan(0,  20,               alpha=0.07, color='red',    label='Buruk (<20)')
    ax1.axhline(y=30, color='orange', linestyle='--', linewidth=1.5)
    ax1.axhline(y=40, color='green',  linestyle='--', linewidth=1.5)
    for qf, psnr in zip(qfs, psnr_vals):
        if qf in [1, 10, 30, 50, 90, 100]:
            ax1.annotate(f'QF={qf}\n{psnr:.1f}dB', (qf, psnr),
                         textcoords='offset points', xytext=(5, 6),
                         fontsize=7.5, color='#1565C0')
    ax1.set_xlabel('Quality Factor')
    ax1.set_ylabel('PSNR (dB)')
    ax1.set_title('PSNR vs Quality Factor\n(Zona warna = kategori kualitas)',
                  fontweight='bold')
    ax1.legend(fontsize=8, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-2, 103)

    # Plot 2: CR vs QF
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(qfs, cr_vals, 'r-s', linewidth=2.5, markersize=8, color='red', zorder=5)
    ax2.fill_between(qfs, cr_vals, 1, where=[c > 1 for c in cr_vals],
                     alpha=0.15, color='red', label='Zona kompresi menguntungkan')
    ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1.5, label='CR = 1')
    for qf, cr in zip(qfs, cr_vals):
        if qf in [1, 10, 30, 50, 90, 100]:
            ax2.annotate(f'QF={qf}\n{cr:.1f}x', (qf, cr),
                         textcoords='offset points', xytext=(5, 6),
                         fontsize=7.5, color='#B71C1C')
    ax2.set_xlabel('Quality Factor')
    ax2.set_ylabel('Compression Ratio')
    ax2.set_title('Compression Ratio vs Quality Factor\n(Semakin tinggi QF = semakin kecil CR)',
                  fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-2, 103)

    # Plot 3: SSIM vs QF
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(qfs, ssim_vals, 'g-^', linewidth=2.5, markersize=8, color='#2E7D32', zorder=5)
    ax3.fill_between(qfs, ssim_vals, alpha=0.15, color='green')
    ax3.axhline(y=0.9, color='orange', linestyle='--', linewidth=1.5, label='SSIM=0.9')
    ax3.axhline(y=0.95, color='green', linestyle='--', linewidth=1.5, label='SSIM=0.95')
    ax3.set_xlabel('Quality Factor')
    ax3.set_ylabel('SSIM')
    ax3.set_title('SSIM vs Quality Factor\n(1.0 = identik dengan asli)',
                  fontweight='bold')
    ax3.set_ylim(0, 1.05)
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(-2, 103)

    # Plot 4: Scatter CR vs PSNR (trade-off utama)
    ax4 = fig.add_subplot(gs[1, 0])
    sc4 = ax4.scatter(cr_vals, psnr_vals, c=qfs, cmap='RdYlGn',
                       s=130, edgecolors='black', linewidth=0.8, zorder=5)
    ax4.plot(cr_vals, psnr_vals, 'k--', alpha=0.25, linewidth=1.5, zorder=4)
    plt.colorbar(sc4, ax=ax4, label='Quality Factor')
    ax4.axhline(y=30, color='orange', linestyle='--', linewidth=1.2, alpha=0.7)
    ax4.axhline(y=40, color='green',  linestyle='--', linewidth=1.2, alpha=0.7)
    for qf, cr, psnr in zip(qfs, cr_vals, psnr_vals):
        if qf in [1, 10, 30, 50, 90]:
            ax4.annotate(f'QF={qf}', (cr, psnr),
                         textcoords='offset points', xytext=(5, 4),
                         fontsize=8.5, color='#1A237E', fontweight='bold')

    # Anotasi daerah optimal
    ax4.annotate('Daerah\nOptimal\n(QF 50-80)',
                  xy=(np.mean(cr_vals[5:9]), np.mean(psnr_vals[5:9])),
                  xytext=(ax4.get_xlim()[0]+0.5 if ax4.get_xlim()[0] > 0 else 1,
                          38),
                  fontsize=9, color='darkgreen', fontweight='bold',
                  arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.5))

    ax4.set_xlabel('Compression Ratio')
    ax4.set_ylabel('PSNR (dB)')
    ax4.set_title('Trade-off Utama: CR vs PSNR\n(Ideal: CR tinggi + PSNR tinggi = pojok kanan atas)',
                  fontweight='bold')
    ax4.grid(True, alpha=0.3)

    # Plot 5: MSE (log scale) vs QF
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.semilogy(qfs, mse_vals, 'm-D', linewidth=2.5, markersize=8, color='purple', zorder=5)
    ax5.fill_between(qfs, mse_vals, 1e-3, alpha=0.10, color='purple')
    for qf, mse in zip(qfs, mse_vals):
        if qf in [1, 10, 30, 50, 90, 100]:
            ax5.annotate(f'{mse:.1f}', (qf, mse),
                         textcoords='offset points', xytext=(5, 6), fontsize=7.5,
                         color='purple')
    ax5.set_xlabel('Quality Factor')
    ax5.set_ylabel('MSE (skala log)')
    ax5.set_title('MSE vs Quality Factor (Skala Log)\n(Semakin rendah = semakin baik)',
                  fontweight='bold')
    ax5.grid(True, alpha=0.3, which='both')
    ax5.set_xlim(-2, 103)

    # Plot 6: Non-zero coefficient % vs QF
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.plot(qfs, nz_pct, 'c-o', linewidth=2.5, markersize=8, color='teal', zorder=5)
    ax6.fill_between(qfs, nz_pct, alpha=0.15, color='teal')

    # Gradient background
    ax6.axhspan(0,  25, alpha=0.06, color='green', label='<25% (sangat terkompresi)')
    ax6.axhspan(25, 60, alpha=0.06, color='yellow', label='25-60% (sedang)')
    ax6.axhspan(60, 100, alpha=0.06, color='red',  label='>60% (sedikit dikompres)')

    for qf, nz in zip(qfs, nz_pct):
        if qf in [1, 10, 30, 50, 90, 100]:
            ax6.annotate(f'{nz:.0f}%', (qf, nz),
                         textcoords='offset points', xytext=(5, 5), fontsize=7.5,
                         color='teal')
    ax6.set_xlabel('Quality Factor')
    ax6.set_ylabel('Koefisien Non-zero (%)')
    ax6.set_title('Persentase Koefisien Non-zero vs QF\n(Lebih sedikit = lebih terkompresi)',
                  fontweight='bold')
    ax6.set_ylim(0, 105)
    ax6.legend(fontsize=7.5)
    ax6.grid(True, alpha=0.3)
    ax6.set_xlim(-2, 103)

    plt.suptitle('Analisis Trade-off Kompresi DCT: Kualitas vs Kompresi',
                 fontsize=14, fontweight='bold')
    plt.savefig('cell9_dct_tradeoff.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_quality_tradeoff(dct_results, quality_list)

# -------------------------------------------------------
# Visualisasi tambahan: Dual axis (CR dan PSNR dalam satu grafik)
# -------------------------------------------------------
qfs_sorted = sorted(dct_results.keys())
psnr_s = [dct_results[qf]['psnr'] for qf in qfs_sorted]
cr_s   = [dct_results[qf]['compression_ratio'] for qf in qfs_sorted]

fig, ax_l = plt.subplots(figsize=(13, 5))
ax_r = ax_l.twinx()

line_psnr, = ax_l.plot(qfs_sorted, psnr_s, 'b-o', linewidth=2.5,
                        markersize=7, label='PSNR (dB)', zorder=5)
ax_l.fill_between(qfs_sorted, psnr_s, alpha=0.10, color='blue')
ax_l.set_xlabel('Quality Factor', fontsize=11)
ax_l.set_ylabel('PSNR (dB)', color='blue', fontsize=11)
ax_l.tick_params(axis='y', labelcolor='blue')
ax_l.axhline(y=30, color='blue', linestyle=':', alpha=0.5)
ax_l.axhline(y=40, color='blue', linestyle=':', alpha=0.5)

line_cr, = ax_r.plot(qfs_sorted, cr_s, 'r-s', linewidth=2.5,
                      markersize=7, label='Compression Ratio', zorder=5)
ax_r.fill_between(qfs_sorted, cr_s, alpha=0.10, color='red')
ax_r.set_ylabel('Compression Ratio', color='red', fontsize=11)
ax_r.tick_params(axis='y', labelcolor='red')

# Tandai titik persilangan optimal
optimal_qf = 50
opt_psnr = dct_results[optimal_qf]['psnr']
opt_cr   = dct_results[optimal_qf]['compression_ratio']
ax_l.axvline(x=optimal_qf, color='purple', linestyle='--',
              linewidth=2, label=f'QF={optimal_qf} (titik keseimbangan)')
ax_l.annotate(f'QF={optimal_qf}\nPSNR={opt_psnr:.1f} dB\nCR={opt_cr:.1f}x',
               xy=(optimal_qf, opt_psnr),
               xytext=(optimal_qf+3, opt_psnr-5),
               fontsize=10, color='purple', fontweight='bold',
               arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))

lines  = [line_psnr, line_cr]
labels = [l.get_label() for l in lines]
ax_l.legend(lines, labels, loc='center right', fontsize=9)

plt.title('Dual Axis: PSNR (Kualitas) vs Compression Ratio per Quality Factor\n'
          'Area Biru = PSNR | Area Merah = CR | Garis Ungu = Titik Keseimbangan',
          fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cell9_dual_axis_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()

print("Analisis trade-off selesai.")

In [ ]:
# Cell 10: Visualisasi mendalam koefisien DCT dan proses kuantisasi

def visualize_dct_coefficients(image, quality_factors=[10, 50, 90]):
    h, w   = image.shape
    cy, cx = h // 2, w // 2
    block  = image[cy:cy+8, cx:cx+8].astype(np.float64)
    block_shifted = block - 128.0
    dct_coeffs    = apply_dct_2d(block_shifted)

    n_cols = len(quality_factors) + 2
    fig, axes = plt.subplots(4, n_cols, figsize=(5 * n_cols, 18))
    fig.patch.set_facecolor('#F5F5F5')

    # ---- Kolom 0: Blok asli ----
    axes[0, 0].imshow(block, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axes[0, 0].set_title('Blok Piksel\nAsli (8x8)', fontweight='bold')
    for i in range(8):
        for j in range(8):
            axes[0, 0].text(j, i, f'{block[i,j]:.0f}', ha='center', va='center',
                             fontsize=8, color='red' if block[i,j] > 180 else 'white')
    axes[0, 0].axis('off')

    # Histogram blok
    axes[1, 0].hist(block.flatten(), bins=16, color='steelblue',
                     edgecolor='navy', alpha=0.8)
    axes[1, 0].set_title('Histogram\nNilai Piksel', fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)

    # Koefisien DCT (log scale)
    log_dct = np.log1p(np.abs(dct_coeffs))
    im20 = axes[2, 0].imshow(log_dct, cmap='inferno', interpolation='nearest')
    axes[2, 0].set_title('Koefisien DCT\n(log scale)', fontweight='bold')
    for i in range(8):
        for j in range(8):
            axes[2, 0].text(j, i, f'{dct_coeffs[i,j]:.0f}', ha='center', va='center',
                             fontsize=7,
                             color='white' if log_dct[i,j] < log_dct.max()*0.6 else 'black')
    axes[2, 0].axis('off')
    plt.colorbar(im20, ax=axes[2, 0], shrink=0.7)

    # Matriks Q standar
    im30 = axes[3, 0].imshow(JPEG_QUANTIZATION_MATRIX, cmap='YlOrRd',
                               vmin=1, vmax=120, interpolation='nearest')
    axes[3, 0].set_title('Matriks\nKuantisasi Dasar', fontweight='bold')
    for i in range(8):
        for j in range(8):
            axes[3, 0].text(j, i, f'{JPEG_QUANTIZATION_MATRIX[i,j]:.0f}',
                             ha='center', va='center', fontsize=8,
                             color='black' if JPEG_QUANTIZATION_MATRIX[i,j] < 70 else 'white')
    axes[3, 0].axis('off')
    plt.colorbar(im30, ax=axes[3, 0], shrink=0.7)

    # ---- Kolom 1: Q Matrix baris ----
    axes[0, 1].axis('off')
    axes[0, 1].text(0.5, 0.5, 'MATRIKS\nKUANTISASI\nPER QF',
                    transform=axes[0, 1].transAxes, ha='center', va='center',
                    fontsize=11, fontweight='bold', color='#1565C0')
    axes[1, 1].axis('off')

    # Energi DCT per posisi frekuensi
    energy_map = dct_coeffs ** 2
    im21 = axes[2, 1].imshow(energy_map, cmap='hot', interpolation='nearest')
    axes[2, 1].set_title('Energi DCT\n(|koef|^2)', fontweight='bold')
    axes[2, 1].axis('off')
    plt.colorbar(im21, ax=axes[2, 1], shrink=0.7)

    # Kurva energi kumulatif (zigzag order)
    zigzag_energy = []
    for i in range(8):
        for j in range(8):
            zigzag_energy.append(dct_coeffs[i, j] ** 2)
    zigzag_energy.sort(reverse=True)
    cumulative_energy = np.cumsum(zigzag_energy) / sum(zigzag_energy) * 100

    axes[3, 1].plot(range(1, 65), cumulative_energy, 'b-o',
                     linewidth=2, markersize=4)
    axes[3, 1].axhline(y=90, color='orange', linestyle='--', linewidth=1.5, label='90%')
    axes[3, 1].axhline(y=99, color='green',  linestyle='--', linewidth=1.5, label='99%')
    axes[3, 1].set_xlabel('Jumlah koefisien (terurut)')
    axes[3, 1].set_ylabel('Energi kumulatif (%)')
    axes[3, 1].set_title('Energi Kumulatif\nKoefisien DCT', fontsize=9, fontweight='bold')
    axes[3, 1].grid(True, alpha=0.3)
    axes[3, 1].legend(fontsize=8)

    # ---- Kolom 2+: Setiap QF ----
    for col_idx, qf in enumerate(quality_factors):
        col      = col_idx + 2
        q_matrix = get_quantization_matrix(qf)
        quantized = quantize_block(dct_coeffs, q_matrix)
        deq       = dequantize_block(quantized, q_matrix)
        recon_blk = np.clip(apply_idct_2d(deq) + 128.0, 0, 255)
        nz_pct    = np.count_nonzero(quantized) / 64 * 100
        block_mse = np.mean((block - recon_blk)**2)

        # Rekonstruksi
        axes[0, col].imshow(recon_blk, cmap='gray', vmin=0, vmax=255,
                             interpolation='nearest')
        axes[0, col].set_title(f'QF = {qf}\nRekonstruksi', fontweight='bold')
        for i in range(8):
            for j in range(8):
                axes[0, col].text(j, i, f'{recon_blk[i,j]:.0f}',
                                   ha='center', va='center', fontsize=8,
                                   color='red' if recon_blk[i,j] > 180 else 'white')
        axes[0, col].axis('off')

        # Matriks Q untuk QF ini
        im1q = axes[1, col].imshow(q_matrix, cmap='YlOrRd', vmin=1, vmax=255,
                                    interpolation='nearest')
        axes[1, col].set_title(f'Q-Matrix QF={qf}\nMin={q_matrix.min():.0f} '
                                f'Max={q_matrix.max():.0f}', fontsize=9, fontweight='bold')
        for i in range(8):
            for j in range(8):
                axes[1, col].text(j, i, f'{q_matrix[i,j]:.0f}',
                                   ha='center', va='center', fontsize=7,
                                   color='black' if q_matrix[i,j] < 150 else 'white')
        axes[1, col].axis('off')
        plt.colorbar(im1q, ax=axes[1, col], shrink=0.7)

        # Kuantisasi result
        max_abs = max(abs(quantized.max()), abs(quantized.min()), 1)
        im2q = axes[2, col].imshow(quantized.astype(float), cmap='RdBu',
                                    vmin=-max_abs, vmax=max_abs,
                                    interpolation='nearest')
        axes[2, col].set_title(f'Koefisien Terkuantisasi\nNon-zero: {nz_pct:.0f}%',
                                fontweight='bold')
        for i in range(8):
            for j in range(8):
                axes[2, col].text(j, i, f'{quantized[i,j]}',
                                   ha='center', va='center', fontsize=8,
                                   color='black' if abs(quantized[i,j]) < max_abs*0.5 else 'white')
        axes[2, col].axis('off')
        plt.colorbar(im2q, ax=axes[2, col], shrink=0.7)

        # Error map
        err_blk = np.abs(block - recon_blk)
        im3q    = axes[3, col].imshow(err_blk, cmap='hot', vmin=0, vmax=30,
                                       interpolation='nearest')
        axes[3, col].set_title(f'Error per Piksel\nMSE={block_mse:.1f}',
                                fontweight='bold')
        for i in range(8):
            for j in range(8):
                axes[3, col].text(j, i, f'{err_blk[i,j]:.0f}',
                                   ha='center', va='center', fontsize=8,
                                   color='white' if err_blk[i,j] > 15 else 'black')
        axes[3, col].axis('off')
        plt.colorbar(im3q, ax=axes[3, col], shrink=0.7)

    row_labels_dct = ['Rekonstruksi Blok', 'Q-Matrix', 'Koefisien Kuantisasi', 'Error Map']
    for row, label in enumerate(row_labels_dct):
        fig.text(0.005, 0.86 - row * 0.215, label, fontsize=9.5,
                 fontweight='bold', rotation=90, va='center', color='#B71C1C')

    plt.suptitle('Analisis Mendalam Koefisien DCT: Pengaruh Quality Factor pada Blok 8x8\n'
                 '(Kolom 1: Data asli | Kolom 2: Energi | Kolom 3+: Per QF)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0.02, 0, 1, 0.96])
    plt.savefig('cell10_dct_coefficients.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_dct_coefficients(test_images['gradient'], quality_factors=[10, 50, 90])

# -------------------------------------------------------
# Visualisasi tambahan: Rekonstruksi bertahap (keep top-k coefficients)
# -------------------------------------------------------
img_blk  = test_images['gradient']
cy2, cx2 = img_blk.shape[0]//2, img_blk.shape[1]//2
blk2     = img_blk[cy2:cy2+8, cx2:cx2+8].astype(np.float64) - 128.0
dct2     = apply_dct_2d(blk2)

keep_list = [1, 2, 4, 8, 16, 32, 48, 64]
fig, axes = plt.subplots(2, len(keep_list), figsize=(22, 6))
fig.patch.set_facecolor('#F8F8F8')

flat_dct = dct2.flatten()
sort_idx = np.argsort(np.abs(flat_dct))[::-1]

for col, k in enumerate(keep_list):
    mask_dct = np.zeros(64)
    mask_dct[sort_idx[:k]] = 1.0
    mask_2d  = mask_dct.reshape(8, 8)
    kept_dct = dct2 * mask_2d
    recon_k  = np.clip(apply_idct_2d(kept_dct) + 128.0, 0, 255)
    mse_k    = np.mean((blk2 + 128 - recon_k)**2)

    axes[0, col].imshow(recon_k, cmap='gray', vmin=0, vmax=255,
                         interpolation='nearest')
    axes[0, col].set_title(f'K={k} koef\nMSE={mse_k:.1f}',
                            fontsize=9, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(mask_2d, cmap='RdYlGn', vmin=0, vmax=1,
                         interpolation='nearest')
    axes[1, col].set_title(f'{k}/64\n({k/64*100:.0f}%)', fontsize=9)
    axes[1, col].axis('off')

fig.text(0.01, 0.75, 'Rekonstruksi', fontsize=10, fontweight='bold',
         rotation=90, va='center')
fig.text(0.01, 0.25, 'Koef yang\ndisimpan', fontsize=10, fontweight='bold',
         rotation=90, va='center')

plt.suptitle('Rekonstruksi Bertahap: Efek Menyimpan K Koefisien DCT Terbesar\n'
             '(Hijau = disimpan | Merah = dibuang | Kiri = sedikit koef | Kanan = semua koef)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0.02, 0, 1, 0.93])
plt.savefig('cell10_dct_progressive.png', dpi=100, bbox_inches='tight')
plt.show()

print("Visualisasi koefisien DCT selesai.")

In [ ]:
# Cell 11: Implementasi Zlib dengan visualisasi level kompresi

def zlib_compress_image(image, level=6):
    raw_bytes       = image.tobytes()
    compressed_data = zlib.compress(raw_bytes, level=level)
    return compressed_data, len(compressed_data)

def zlib_decompress_image(compressed_data, shape, dtype=np.uint8):
    raw_bytes   = zlib.decompress(compressed_data)
    image_array = np.frombuffer(raw_bytes, dtype=dtype)
    return image_array.reshape(shape)

def compare_compression_levels(image, image_name="Citra"):
    original_bytes = image.nbytes
    results = {}
    print(f"\nKompresi Zlib: {image_name} | Asli: {original_bytes:,} bytes")
    print("=" * 72)
    print(f"{'Level':>7} {'Bytes':>10} {'CR':>8} {'Hemat(%)':>10} {'Waktu(ms)':>12}")
    print("-" * 72)
    for level in range(0, 10):
        t0 = time.time()
        comp, comp_bytes = zlib_compress_image(image, level=level)
        elapsed = (time.time() - t0) * 1000
        recon   = zlib_decompress_image(comp, image.shape)
        cr      = calculate_compression_ratio(original_bytes, comp_bytes)
        hemat   = (1 - comp_bytes / original_bytes) * 100
        results[level] = {
            'compressed_bytes': comp_bytes, 'compression_ratio': cr,
            'space_saved': hemat, 'time_ms': elapsed,
            'lossless': np.array_equal(image, recon)
        }
        print(f"{level:>7} {comp_bytes:>10,} {cr:>8.3f} {hemat:>9.1f}% {elapsed:>11.2f}")
    print("=" * 72)
    return results

# Jalankan untuk semua citra
all_zlib_results = {}
for img_name in test_images.keys():
    all_zlib_results[img_name] = compare_compression_levels(
        test_images[img_name], img_name
    )

# -------------------------------------------------------
# Visualisasi 1: Dashboard level kompresi Zlib
# -------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#FAFAFA')

img_names_z = list(test_images.keys())
levels_z    = list(range(10))

# Plot CR per level untuk setiap citra
ax = axes[0, 0]
color_map_z = plt.cm.tab10(np.linspace(0, 1, len(img_names_z)))
for i, name in enumerate(img_names_z):
    cr_per_level = [all_zlib_results[name][lv]['compression_ratio'] for lv in levels_z]
    ax.plot(levels_z, cr_per_level, '-o', linewidth=2, markersize=6,
            color=color_map_z[i], label=name)
ax.set_xlabel('Level Kompresi')
ax.set_ylabel('Compression Ratio')
ax.set_title('CR vs Level Kompresi\n(per jenis citra)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xticks(levels_z)

# Plot waktu kompresi
ax2 = axes[0, 1]
for i, name in enumerate(img_names_z):
    time_per_level = [all_zlib_results[name][lv]['time_ms'] for lv in levels_z]
    ax2.plot(levels_z, time_per_level, '-s', linewidth=2, markersize=6,
             color=color_map_z[i], label=name)
ax2.set_xlabel('Level Kompresi')
ax2.set_ylabel('Waktu Kompresi (ms)')
ax2.set_title('Waktu Kompresi vs Level\n(Level tinggi = lebih lambat)',
              fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(levels_z)

# Plot heatmap CR (citra x level)
ax3 = axes[0, 2]
heatmap_data = np.array([
    [all_zlib_results[name][lv]['compression_ratio'] for lv in levels_z]
    for name in img_names_z
])
im3 = ax3.imshow(heatmap_data, cmap='RdYlGn', aspect='auto',
                   vmin=0.5, vmax=heatmap_data.max())
ax3.set_xticks(levels_z)
ax3.set_xticklabels([str(lv) for lv in levels_z])
ax3.set_yticks(range(len(img_names_z)))
ax3.set_yticklabels(img_names_z)
ax3.set_xlabel('Level Kompresi')
ax3.set_title('Heatmap Compression Ratio\n(Hijau=baik, Merah=kurang baik)',
              fontweight='bold')
for i in range(len(img_names_z)):
    for j in levels_z:
        ax3.text(j, i, f'{heatmap_data[i,j]:.2f}',
                  ha='center', va='center', fontsize=7.5,
                  color='black' if heatmap_data[i,j] < 2 else 'white')
plt.colorbar(im3, ax=ax3, shrink=0.8, label='CR')

# Plot penghematan ruang level 6 (default)
ax4 = axes[1, 0]
hemat_l6 = [all_zlib_results[name][6]['space_saved'] for name in img_names_z]
bar_c4   = ['#4CAF50' if h > 0 else '#EF5350' for h in hemat_l6]
bars4    = ax4.bar(img_names_z, hemat_l6, color=bar_c4,
                    edgecolor='black', linewidth=0.8)
ax4.axhline(y=0, color='black', linestyle='--', linewidth=1.5)
ax4.set_ylabel('Penghematan Ruang (%)')
ax4.set_title('Penghematan Ruang Zlib Level 6 (Default)\nper Jenis Citra',
              fontweight='bold')
ax4.tick_params(axis='x', rotation=20)
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars4, hemat_l6):
    ypos = bar.get_height() + 0.5 if val >= 0 else bar.get_height() - 3
    ax4.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Plot scatter CR vs Waktu
ax5 = axes[1, 1]
for i, name in enumerate(img_names_z):
    cr_pts   = [all_zlib_results[name][lv]['compression_ratio'] for lv in levels_z]
    time_pts = [all_zlib_results[name][lv]['time_ms']           for lv in levels_z]
    sc5 = ax5.scatter(time_pts, cr_pts, c=levels_z, cmap='coolwarm',
                       s=80, edgecolors='black', linewidth=0.5, alpha=0.8)
    ax5.plot(time_pts, cr_pts, '--', color=color_map_z[i], alpha=0.4, linewidth=1)

plt.colorbar(sc5, ax=ax5, label='Level', shrink=0.8)
ax5.set_xlabel('Waktu Kompresi (ms)')
ax5.set_ylabel('Compression Ratio')
ax5.set_title('Trade-off: Waktu vs CR\n(Level rendah=cepat-CR rendah, '
              'Level tinggi=lambat-CR tinggi)', fontweight='bold')
ax5.grid(True, alpha=0.3)

# Verifikasi lossless
ax6 = axes[1, 2]
ax6.axis('off')
ax6.set_facecolor('#E8F5E9')

lossless_text = "VERIFIKASI LOSSLESS ZLIB\n" + "=" * 30 + "\n"
for name in img_names_z:
    for lv in [0, 3, 6, 9]:
        is_ll = all_zlib_results[name][lv]['lossless']
        status = "PASS" if is_ll else "FAIL"
        lossless_text += f"{name} (L{lv}): {status}\n"

lossless_text += "\n" + "=" * 30 + "\n"
lossless_text += "Semua level Zlib bersifat\nLOSSLESS (100% identik)"

ax6.text(0.05, 0.97, lossless_text, transform=ax6.transAxes,
         fontsize=9.5, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#C8E6C9', alpha=0.9))

plt.suptitle('Analisis Kompresi Zlib: Level, Waktu, dan Efektivitas',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cell11_zlib_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Perbandingan ukuran file (bar horizontal)
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 7))
y_pos      = np.arange(len(img_names_z))
bar_height = 0.12
offsets    = np.linspace(-0.4, 0.4, 10)
cmap_zlib  = plt.cm.plasma(np.linspace(0.1, 0.9, 10))

for lv in range(10):
    comp_bytes = [all_zlib_results[name][lv]['compressed_bytes'] for name in img_names_z]
    ax.barh(y_pos + offsets[lv], comp_bytes, bar_height,
            color=cmap_zlib[lv], alpha=0.85, label=f'L{lv}',
            edgecolor='none')

orig_bytes = [test_images[name].nbytes for name in img_names_z]
ax.barh(y_pos, orig_bytes, bar_height * 0.5,
        color='black', alpha=0.4, label='Asli', edgecolor='black', linewidth=1.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(img_names_z)
ax.set_xlabel('Ukuran File (bytes)')
ax.set_title('Perbandingan Ukuran File: Setiap Level Kompresi Zlib\n'
             '(Hitam = ukuran asli | Warna = ukuran terkompresi per level)',
             fontweight='bold')
ax.legend(ncol=6, fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
ax.axvline(x=test_images['gradient'].nbytes, color='black',
            linestyle='--', linewidth=1.5, alpha=0.4, label='Ukuran asli')

plt.tight_layout()
plt.savefig('cell11_zlib_file_sizes.png', dpi=100, bbox_inches='tight')
plt.show()

print("Analisis Zlib selesai. Semua level bersifat lossless.")

In [ ]:
# Cell 12: Perbandingan komprehensif semua metode + visualisasi

def comprehensive_comparison(images_dict, quality_factor=50):
    methods = {
        'RLE':    {'type': 'lossless', 'color': '#1565C0'},
        'Zlib-1': {'type': 'lossless', 'color': '#2E7D32'},
        'Zlib-6': {'type': 'lossless', 'color': '#558B2F'},
        'Zlib-9': {'type': 'lossless', 'color': '#9E9D24'},
        'DCT-90': {'type': 'lossy',    'color': '#FF8F00'},
        'DCT-50': {'type': 'lossy',    'color': '#E64A19'},
        'DCT-10': {'type': 'lossy',    'color': '#B71C1C'},
    }
    all_results = {}

    for img_name, image in images_dict.items():
        orig_b  = image.nbytes
        img_res = {}

        enc, shp, _ = rle_encode_image(image)
        rle_rec     = rle_decode_image(enc, shp)
        rle_b       = calculate_rle_size(enc)
        img_res['RLE'] = {
            'compression_ratio': calculate_compression_ratio(orig_b, rle_b),
            'psnr': calculate_psnr(image, rle_rec),
            'ssim': calculate_ssim(image, rle_rec),
            'compressed_bytes': rle_b,
            'reconstructed': rle_rec
        }

        for lvl, key in [(1,'Zlib-1'),(6,'Zlib-6'),(9,'Zlib-9')]:
            comp_d, comp_b = zlib_compress_image(image, level=lvl)
            zlib_rec       = zlib_decompress_image(comp_d, image.shape)
            img_res[key]   = {
                'compression_ratio': calculate_compression_ratio(orig_b, comp_b),
                'psnr': calculate_psnr(image, zlib_rec),
                'ssim': calculate_ssim(image, zlib_rec),
                'compressed_bytes': comp_b,
                'reconstructed': zlib_rec
            }

        h, w = image.shape
        for qf, key in [(90,'DCT-90'),(50,'DCT-50'),(10,'DCT-10')]:
            qblk, qmat, pshp, nz = dct_compress_image(image, quality_factor=qf)
            dct_rec = dct_decompress_image(qblk, qmat, image.shape)
            dct_b   = estimate_dct_compressed_size(nz, (h//8+1)*(w//8+1))
            img_res[key] = {
                'compression_ratio': calculate_compression_ratio(orig_b, dct_b),
                'psnr': calculate_psnr(image, dct_rec),
                'ssim': calculate_ssim(image, dct_rec),
                'compressed_bytes': dct_b,
                'reconstructed': dct_rec
            }

        all_results[img_name] = img_res

    return all_results, methods

print("Menjalankan perbandingan komprehensif ...")
comp_results, method_info = comprehensive_comparison(test_images)

# Cetak tabel
for img_name in test_images.keys():
    print(f"\nCitra: {img_name.upper()}")
    print(f"{'Metode':<12} {'Jenis':<10} {'CR':>8} {'PSNR':>10} {'SSIM':>8}")
    print("-" * 55)
    for method, info in method_info.items():
        r = comp_results[img_name][method]
        p = f"{r['psnr']:.2f}" if r['psnr'] != float('inf') else "inf"
        print(f"{method:<12} {info['type']:<10} {r['compression_ratio']:>8.2f} "
              f"{p:>10} {r['ssim']:>8.4f}")
    print("-" * 55)

# -------------------------------------------------------
# Visualisasi 1: Grid rekonstruksi semua metode x semua citra
# -------------------------------------------------------
methods_list = list(method_info.keys())
img_list     = list(test_images.keys())
n_methods    = len(methods_list)
n_imgs       = len(img_list)

fig, axes = plt.subplots(n_methods + 1, n_imgs, figsize=(n_imgs*4, (n_methods+1)*3.8))
fig.patch.set_facecolor('#EEEEEE')

# Baris pertama: citra asli
for col, img_name in enumerate(img_list):
    axes[0, col].imshow(test_images[img_name], cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f'{img_name.upper()}\n(Asli)', fontweight='bold', fontsize=9)
    axes[0, col].axis('off')

# Baris berikutnya: setiap metode
for row, method in enumerate(methods_list):
    info   = method_info[method]
    method_type_color = '#1565C0' if info['type'] == 'lossless' else '#B71C1C'

    for col, img_name in enumerate(img_list):
        res   = comp_results[img_name][method]
        recon = res['reconstructed']
        psnr  = res['psnr']
        cr    = res['compression_ratio']

        psnr_str = 'inf' if psnr == float('inf') else f"{psnr:.1f}"

        axes[row+1, col].imshow(recon, cmap='gray', vmin=0, vmax=255)
        axes[row+1, col].set_title(
            f'PSNR:{psnr_str} CR:{cr:.1f}x',
            fontsize=8
        )
        axes[row+1, col].axis('off')

        # Warna border berdasarkan kualitas
        for spine in axes[row+1, col].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
            if psnr == float('inf') or psnr >= 40:
                spine.set_edgecolor('#2E7D32')
            elif psnr >= 30:
                spine.set_edgecolor('#F57F17')
            else:
                spine.set_edgecolor('#B71C1C')

    # Label metode di kiri
    fig.text(0.005, 1 - (row + 1.5) / (n_methods + 1),
             f'{method}\n({info["type"]})', fontsize=9.5,
             fontweight='bold', va='center', color=method_type_color)

plt.suptitle('Grid Rekonstruksi: Semua Metode x Semua Citra\n'
             '(Border Hijau=PSNR>40dB | Kuning=30-40dB | Merah=<30dB)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0.04, 0, 1, 0.97])
plt.savefig('cell12_reconstruction_grid.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 13: Visualisasi perbandingan komprehensif semua metode

def plot_comprehensive_comparison(comp_results, method_info, images_dict):
    methods   = list(method_info.keys())
    img_names = list(comp_results.keys())
    colors    = [method_info[m]['color'] for m in methods]

    fig = plt.figure(figsize=(20, 20))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.32)

    # ---- Plot 1: CR grouped bars ----
    ax1   = fig.add_subplot(gs[0, 0])
    x     = np.arange(len(img_names))
    n_m   = len(methods)
    bw    = 0.10
    for i, (method, color) in enumerate(zip(methods, colors)):
        cr_vals = [comp_results[img][method]['compression_ratio'] for img in img_names]
        offset  = (i - n_m / 2) * bw + bw / 2
        ax1.bar(x + offset, cr_vals, bw, label=method, color=color,
                edgecolor='black', linewidth=0.4, alpha=0.9)
    ax1.axhline(y=1, color='black', linestyle='--', linewidth=2, label='CR=1')
    ax1.set_xticks(x)
    ax1.set_xticklabels(img_names, rotation=20, ha='right')
    ax1.set_ylabel('Compression Ratio')
    ax1.set_title('Compression Ratio: Semua Metode x Semua Citra', fontweight='bold')
    ax1.legend(fontsize=7, ncol=2)
    ax1.grid(True, alpha=0.3, axis='y')

    # ---- Plot 2: PSNR grouped bars ----
    ax2 = fig.add_subplot(gs[0, 1])
    for i, (method, color) in enumerate(zip(methods, colors)):
        psnr_vals = []
        for img in img_names:
            v = comp_results[img][method]['psnr']
            psnr_vals.append(52 if v == float('inf') else v)
        offset = (i - n_m / 2) * bw + bw / 2
        ax2.bar(x + offset, psnr_vals, bw, label=method, color=color,
                edgecolor='black', linewidth=0.4, alpha=0.9)
    ax2.axhline(y=30, color='orange', linestyle='--', linewidth=1.5, label='30 dB')
    ax2.axhline(y=40, color='green',  linestyle='--', linewidth=1.5, label='40 dB')
    ax2.axhline(y=52, color='gray',   linestyle=':',  linewidth=1.5, label='52=inf (lossless)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(img_names, rotation=20, ha='right')
    ax2.set_ylabel('PSNR (dB)')
    ax2.set_title('PSNR: Semua Metode x Semua Citra\n(52 dB = nilai "infiniti" / lossless)',
                  fontweight='bold')
    ax2.legend(fontsize=7, ncol=2)
    ax2.grid(True, alpha=0.3, axis='y')

    # ---- Plot 3: Scatter PSNR vs CR ----
    ax3 = fig.add_subplot(gs[1, 0])
    marker_map = {
        'RLE':'o','Zlib-1':'s','Zlib-6':'D','Zlib-9':'^',
        'DCT-90':'*','DCT-50':'P','DCT-10':'X'
    }
    for method, color in zip(methods, colors):
        cr_pts   = [comp_results[img][method]['compression_ratio'] for img in img_names]
        psnr_pts = [comp_results[img][method]['psnr']              for img in img_names]
        psnr_pts = [52 if p == float('inf') else p for p in psnr_pts]
        ax3.scatter(cr_pts, psnr_pts, c=color, label=method,
                    s=160, edgecolors='black', linewidth=0.8,
                    marker=marker_map[method], zorder=5, alpha=0.9)

    ax3.axhline(y=30, color='orange', linestyle='--', linewidth=1.2, alpha=0.6)
    ax3.axhline(y=40, color='green',  linestyle='--', linewidth=1.2, alpha=0.6)
    ax3.axhspan(40, 55, alpha=0.05, color='green')
    ax3.axhspan(30, 40, alpha=0.05, color='yellow')
    ax3.axhspan(0,  30, alpha=0.05, color='red')
    ax3.set_xlabel('Compression Ratio')
    ax3.set_ylabel('PSNR (dB)')
    ax3.set_title('Trade-off Utama: CR vs PSNR\n(Ideal = pojok kanan atas)',
                  fontweight='bold')
    ax3.legend(fontsize=8, ncol=2)
    ax3.grid(True, alpha=0.3)

    # Anotasi arah ideal
    ax3.annotate('', xy=(ax3.get_xlim()[1]*0.95, 50),
                  xytext=(ax3.get_xlim()[1]*0.6, 50),
                  arrowprops=dict(arrowstyle='->', color='darkblue', lw=2.5))
    ax3.text(ax3.get_xlim()[1]*0.65, 51, 'Ideal', color='darkblue', fontweight='bold')

    # ---- Plot 4: Heatmap SSIM ----
    ax4 = fig.add_subplot(gs[1, 1])
    ssim_matrix = np.array([
        [comp_results[img][m]['ssim'] for m in methods]
        for img in img_names
    ])
    im4 = ax4.imshow(ssim_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax4.set_xticks(range(len(methods)))
    ax4.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
    ax4.set_yticks(range(len(img_names)))
    ax4.set_yticklabels(img_names)
    ax4.set_title('Heatmap SSIM (Citra x Metode)\n(Hijau=1.0/sempurna, Merah=buruk)',
                  fontweight='bold')
    plt.colorbar(im4, ax=ax4, shrink=0.8, label='SSIM')
    for i in range(len(img_names)):
        for j in range(len(methods)):
            val = ssim_matrix[i, j]
            ax4.text(j, i, f'{val:.3f}', ha='center', va='center',
                     fontsize=8, fontweight='bold',
                     color='black' if 0.3 < val < 0.8 else 'white')

    # ---- Plot 5: Rata-rata CR per metode (barh) ----
    ax5 = fig.add_subplot(gs[2, 0])
    avg_cr   = [np.mean([comp_results[img][m]['compression_ratio'] for img in img_names])
                for m in methods]
    avg_ssim = [np.mean([comp_results[img][m]['ssim']              for img in img_names])
                for m in methods]

    type_labels = [f"{m}\n({method_info[m]['type']})" for m in methods]
    bars5 = ax5.barh(type_labels, avg_cr, color=colors, edgecolor='black',
                      linewidth=0.8, alpha=0.9)
    ax5.set_xlabel('Rata-rata Compression Ratio')
    ax5.set_title('Rata-rata CR per Metode\n(Lintas semua citra uji)', fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='x')
    for bar, val, ss in zip(bars5, avg_cr, avg_ssim):
        ax5.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'CR={val:.2f} | SSIM={ss:.3f}',
                 va='center', fontsize=9)

    # ---- Plot 6: Radar/Spider per citra ----
    ax6 = fig.add_subplot(gs[2, 1])
    categories  = ['CR\n(norm)', 'PSNR\n(norm)', 'SSIM']
    n_cat       = len(categories)

    all_cr_flat   = [comp_results[img][m]['compression_ratio']
                     for img in img_names for m in methods]
    all_psnr_flat = [min(comp_results[img][m]['psnr'], 52)
                     for img in img_names for m in methods]
    max_cr   = max(all_cr_flat)
    max_psnr = 52

    cm6 = plt.cm.tab10(np.linspace(0, 1, len(methods)))
    for i, (method, color6) in enumerate(zip(methods, cm6)):
        norm_cr   = np.mean([comp_results[img][method]['compression_ratio']
                             for img in img_names]) / max_cr
        norm_psnr = np.mean([min(comp_results[img][method]['psnr'], 52)
                             for img in img_names]) / max_psnr
        norm_ssim = np.mean([comp_results[img][method]['ssim'] for img in img_names])

        x_vals = [0, 1, 2]
        y_vals = [norm_cr, norm_psnr, norm_ssim]

        ax6.plot(x_vals, y_vals, '-o', linewidth=2, markersize=8,
                  color=color6, label=method, alpha=0.85)
        ax6.fill_between(x_vals, y_vals, alpha=0.07, color=color6)

    ax6.set_xticks([0, 1, 2])
    ax6.set_xticklabels(categories)
    ax6.set_ylim(0, 1.05)
    ax6.set_ylabel('Nilai Ternormalisasi (0-1)')
    ax6.set_title('Profil Kinerja Ternormalisasi\n(CR, PSNR, SSIM dinormalisasi ke [0,1])',
                  fontweight='bold')
    ax6.legend(fontsize=8, ncol=2, loc='lower right')
    ax6.grid(True, alpha=0.3)
    ax6.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    plt.suptitle('Dashboard Komprehensif: Perbandingan Semua Metode Kompresi Citra',
                 fontsize=14, fontweight='bold')
    plt.savefig('cell13_comprehensive_dashboard.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_comprehensive_comparison(comp_results, method_info, test_images)
print("Perbandingan komprehensif selesai.")

In [ ]:
# Cell 14: Analisis artefak kompresi dengan visualisasi mendalam

def analyze_compression_artifacts(image, quality_factors=[5, 20, 50]):
    fig, axes = plt.subplots(5, len(quality_factors)+1,
                              figsize=(5*(len(quality_factors)+1), 22))
    fig.patch.set_facecolor('#F0F0F0')

    cy, cx      = image.shape[0]//2, image.shape[1]//2
    crop_size   = 64
    row_mid     = image.shape[0]//2

    def get_crop(img):
        return img[cy-crop_size//2:cy+crop_size//2,
                   cx-crop_size//2:cx+crop_size//2]

    # Kolom 0: Asli
    axes[0,0].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[0,0].set_title('Citra Asli\n(Original)', fontweight='bold', fontsize=10)
    axes[0,0].axis('off')

    crop_orig = get_crop(image)
    crop_big  = cv2.resize(crop_orig, (192, 192), interpolation=cv2.INTER_NEAREST)
    axes[1,0].imshow(crop_big, cmap='gray', vmin=0, vmax=255)
    axes[1,0].set_title(f'Detail Tengah\n({crop_size}x{crop_size})', fontsize=9)
    axes[1,0].axis('off')

    axes[2,0].plot(image[row_mid,:], 'b-', linewidth=1.5, label='Asli')
    axes[2,0].set_title('Profil Baris Tengah', fontsize=9, fontweight='bold')
    axes[2,0].set_xlabel('Kolom')
    axes[2,0].set_ylabel('Nilai Piksel')
    axes[2,0].set_xlim(0, image.shape[1])
    axes[2,0].set_ylim(-5, 265)
    axes[2,0].grid(True, alpha=0.3)
    axes[2,0].legend(fontsize=8)

    axes[3,0].hist(image.flatten(), bins=64, color='#1565C0',
                    edgecolor='navy', alpha=0.8, linewidth=0.5)
    axes[3,0].set_title('Histogram Piksel\n(Asli)', fontsize=9, fontweight='bold')
    axes[3,0].set_xlabel('Nilai Piksel')
    axes[3,0].set_ylabel('Frekuensi')
    axes[3,0].set_xlim(0, 255)
    axes[3,0].grid(True, alpha=0.3)

    # Gradien untuk mendeteksi tepi
    grad_orig = np.abs(np.gradient(image.astype(float), axis=1))
    axes[4,0].imshow(grad_orig, cmap='hot', vmin=0, vmax=grad_orig.max())
    axes[4,0].set_title('Peta Gradien\n(Deteksi Tepi)', fontsize=9, fontweight='bold')
    axes[4,0].axis('off')

    for col_idx, qf in enumerate(quality_factors):
        col = col_idx + 1

        qblk, qmat, pshp, nz = dct_compress_image(image, quality_factor=qf)
        recon = dct_decompress_image(qblk, qmat, image.shape)

        psnr_val = calculate_psnr(image, recon)
        ssim_val = calculate_ssim(image, recon)
        cr_val   = calculate_compression_ratio(image.nbytes,
                    estimate_dct_compressed_size(nz, (image.shape[0]//8+1)*(image.shape[1]//8+1)))

        qf_color = '#2E7D32' if psnr_val>=40 else '#F57F17' if psnr_val>=30 else '#B71C1C'

        # Rekonstruksi dengan grid blok 8x8
        axes[0,col].imshow(recon, cmap='gray', vmin=0, vmax=255)
        if qf <= 20:
            for gi in range(0, image.shape[0], 8):
                axes[0,col].axhline(y=gi, color='red', linewidth=0.4, alpha=0.6)
            for gj in range(0, image.shape[1], 8):
                axes[0,col].axvline(x=gj, color='red', linewidth=0.4, alpha=0.6)
        axes[0,col].set_title(f'QF = {qf}\nCR: {cr_val:.1f}x',
                               fontweight='bold', fontsize=10, color=qf_color)
        axes[0,col].axis('off')

        # Detail crop (diperbesar)
        crop_recon = get_crop(recon)
        crop_recon_big = cv2.resize(crop_recon, (192, 192), interpolation=cv2.INTER_NEAREST)
        axes[1,col].imshow(crop_recon_big, cmap='gray', vmin=0, vmax=255)
        # Grid pada crop
        block_px = 192 // crop_size * 8
        for gi in range(0, 192, block_px):
            axes[1,col].axhline(y=gi, color='red', linewidth=0.7, alpha=0.7)
            axes[1,col].axvline(x=gi, color='red', linewidth=0.7, alpha=0.7)
        axes[1,col].set_title(f'Detail (x{192//crop_size})\nSSIM: {ssim_val:.4f}',
                               fontsize=9, color=qf_color)
        axes[1,col].axis('off')

        # Profil baris
        axes[2,col].plot(image[row_mid,:], 'b-', linewidth=1, alpha=0.5, label='Asli')
        axes[2,col].plot(recon[row_mid,:],  'r-', linewidth=1.5, label=f'QF={qf}')
        axes[2,col].fill_between(range(image.shape[1]),
                                   image[row_mid,:].astype(float),
                                   recon[row_mid,:].astype(float),
                                   alpha=0.25, color='red', label='Error area')
        axes[2,col].set_title(f'Profil Baris\nQF={qf}', fontsize=9, fontweight='bold')
        axes[2,col].set_xlabel('Kolom')
        axes[2,col].set_ylabel('Nilai Piksel')
        axes[2,col].set_xlim(0, image.shape[1])
        axes[2,col].set_ylim(-5, 265)
        axes[2,col].grid(True, alpha=0.3)
        axes[2,col].legend(fontsize=7)
        # Tandai batas blok 8x8 pada profil
        for bnd in range(0, image.shape[1], 8):
            axes[2,col].axvline(x=bnd, color='gray', linewidth=0.4, alpha=0.4)

        # Histogram rekonstruksi + overlay asli
        axes[3,col].hist(image.flatten(), bins=64, color='blue',
                          alpha=0.3, label='Asli', density=True)
        axes[3,col].hist(recon.flatten(), bins=64, color='red',
                          alpha=0.5, label=f'QF={qf}', density=True)
        axes[3,col].set_title(f'Histogram\nPSNR={psnr_val:.1f} dB', fontsize=9,
                               fontweight='bold', color=qf_color)
        axes[3,col].set_xlabel('Nilai Piksel')
        axes[3,col].set_xlim(0, 255)
        axes[3,col].legend(fontsize=7)
        axes[3,col].grid(True, alpha=0.3)

        # Peta gradien rekonstruksi
        grad_recon = np.abs(np.gradient(recon.astype(float), axis=1))
        axes[4,col].imshow(grad_recon, cmap='hot', vmin=0, vmax=grad_orig.max())
        axes[4,col].set_title(f'Peta Gradien\n(Artefak blok terlihat jelas QF rendah)',
                               fontsize=8, fontweight='bold')
        axes[4,col].axis('off')

    row_names = ['Citra Full', 'Detail (zoom)', 'Profil Baris', 'Histogram', 'Peta Gradien']
    for row, rname in enumerate(row_names):
        fig.text(0.005, 0.90 - row * 0.175, rname, fontsize=9.5,
                 fontweight='bold', rotation=90, va='center', color='#1A237E')

    plt.suptitle('Analisis Artefak Kompresi DCT\n'
                 '(Garis merah pada citra dan crop = batas blok 8x8 | '
                 'Hijau=PSNR>40 | Kuning=30-40 | Merah=<30)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0.02, 0, 1, 0.97])
    plt.savefig('cell14_artifacts_analysis.png', dpi=100, bbox_inches='tight')
    plt.show()

print("Analisis artefak kompresi DCT:")
analyze_compression_artifacts(test_images['geometric'], quality_factors=[5, 20, 50])

# -------------------------------------------------------
# Visualisasi tambahan: Deteksi blocking artifact kuantitatif
# -------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
qf_test = [5, 20, 50, 90]

for col, qf in enumerate(qf_test):
    img_ref = test_images['geometric']
    qblk, qmat, _, _ = dct_compress_image(img_ref, quality_factor=qf)
    recon = dct_decompress_image(qblk, qmat, img_ref.shape)

    # Hitung perbedaan antar blok (blocking metric)
    block_diff_h = np.zeros_like(img_ref, dtype=float)
    block_diff_v = np.zeros_like(img_ref, dtype=float)
    for bi in range(8, img_ref.shape[0], 8):
        block_diff_h[bi, :] = np.abs(
            recon[bi,:].astype(float) - recon[bi-1,:].astype(float)
        )
    for bj in range(8, img_ref.shape[1], 8):
        block_diff_v[:, bj] = np.abs(
            recon[:,bj].astype(float) - recon[:,bj-1].astype(float)
        )
    blocking_map = block_diff_h + block_diff_v

    im = axes[col].imshow(blocking_map, cmap='hot', vmin=0, vmax=30)
    axes[col].set_title(f'QF = {qf}\nMean Blocking: {blocking_map.mean():.2f}',
                         fontweight='bold')
    axes[col].axis('off')
    plt.colorbar(im, ax=axes[col], shrink=0.75)

plt.suptitle('Peta Blocking Artifact Kuantitatif (Perbedaan Nilai di Batas Blok)\n'
             '(Putih/Merah = perbedaan besar = artefak kuat)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cell14_blocking_metric.png', dpi=100, bbox_inches='tight')
plt.show()

print("Analisis artefak selesai.")

In [ ]:
# Cell 15: PNG vs JPEG dengan visualisasi lengkap

def save_and_compare_formats(image, filename_base='comparison'):
    results    = {}
    raw_bytes  = image.nbytes

    cv2.imwrite(f'{filename_base}_lossless.png', image,
                [cv2.IMWRITE_PNG_COMPRESSION, 9])
    png_bytes = os.path.getsize(f'{filename_base}_lossless.png')
    png_img   = cv2.imread(f'{filename_base}_lossless.png', cv2.IMREAD_GRAYSCALE)
    results['PNG (Lossless)'] = {
        'path': f'{filename_base}_lossless.png',
        'bytes': png_bytes,
        'compression_ratio': calculate_compression_ratio(raw_bytes, png_bytes),
        'psnr': calculate_psnr(image, png_img),
        'ssim': calculate_ssim(image, png_img),
        'reconstructed': png_img, 'type': 'lossless'
    }

    for quality in [90, 50, 10]:
        path = f'{filename_base}_q{quality}.jpg'
        cv2.imwrite(path, image, [cv2.IMWRITE_JPEG_QUALITY, quality])
        jpeg_bytes = os.path.getsize(path)
        jpeg_img   = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        results[f'JPEG Q={quality}'] = {
            'path': path,
            'bytes': jpeg_bytes,
            'compression_ratio': calculate_compression_ratio(raw_bytes, jpeg_bytes),
            'psnr': calculate_psnr(image, jpeg_img),
            'ssim': calculate_ssim(image, jpeg_img),
            'reconstructed': jpeg_img, 'type': 'lossy'
        }

    print(f"\nUkuran RAW (tanpa kompresi): {raw_bytes:,} bytes")
    print("=" * 68)
    print(f"{'Format':<16} {'Bytes':>10} {'CR':>7} {'PSNR':>9} {'SSIM':>7} {'Jenis':>9}")
    print("-" * 68)
    for fmt, d in results.items():
        p = f"{d['psnr']:.2f}" if d['psnr'] != float('inf') else "inf"
        print(f"{fmt:<16} {d['bytes']:>10,} {d['compression_ratio']:>7.2f} "
              f"{p:>9} {d['ssim']:>7.4f} {d['type']:>9}")
    print("=" * 68)
    return results, raw_bytes

print("Menyimpan citra dalam berbagai format...")
format_results, raw_size = save_and_compare_formats(
    test_images['geometric'], 'cell15_comparison'
)

# -------------------------------------------------------
# Visualisasi 1: Strip rekonstruksi + error + histogram
# -------------------------------------------------------
fmt_names = list(format_results.keys())
n_fmt     = len(fmt_names)

fig, axes = plt.subplots(4, n_fmt, figsize=(n_fmt*4.5, 17))
fig.patch.set_facecolor('#F0F0F0')

ref_img = test_images['geometric']
for col, (fmt_name, data) in enumerate(format_results.items()):
    img_saved = data['reconstructed']
    psnr_val  = data['psnr']
    cr_val    = data['compression_ratio']
    ssim_val  = data['ssim']
    psnr_str  = f"{psnr_val:.1f}" if psnr_val != float('inf') else "inf"
    tc = '#2E7D32' if data['type'] == 'lossless' else '#B71C1C'

    # Rekonstruksi
    axes[0, col].imshow(img_saved, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f"{fmt_name}\n{data['bytes']:,} bytes\nCR: {cr_val:.2f}x",
                            fontsize=9, fontweight='bold', color=tc)
    axes[0, col].axis('off')

    # Error map
    err = np.abs(ref_img.astype(np.int32) - img_saved.astype(np.int32))
    im2 = axes[1, col].imshow(err, cmap='hot', vmin=0, vmax=30)
    axes[1, col].set_title(f'PSNR: {psnr_str} dB\nSSIM: {ssim_val:.4f}',
                            fontsize=9, color=tc)
    axes[1, col].axis('off')
    plt.colorbar(im2, ax=axes[1, col], shrink=0.7)

    # Histogram overlay
    axes[2, col].hist(ref_img.flatten(), bins=64, color='blue',
                       alpha=0.4, label='Asli', density=True)
    axes[2, col].hist(img_saved.flatten(), bins=64, color='red',
                       alpha=0.5, label=fmt_name.split()[0], density=True)
    axes[2, col].set_title('Histogram\nAsli vs Rekonstruksi', fontsize=8)
    axes[2, col].set_xlabel('Nilai Piksel', fontsize=8)
    axes[2, col].legend(fontsize=7)
    axes[2, col].grid(True, alpha=0.3)
    axes[2, col].set_xlim(0, 255)

    # Profil baris tengah
    row_m = ref_img.shape[0]//2
    axes[3, col].plot(ref_img[row_m,:], 'b-', linewidth=1.5, label='Asli', alpha=0.7)
    axes[3, col].plot(img_saved[row_m,:], 'r-', linewidth=1.5, label='Rekonstruksi', alpha=0.7)
    axes[3, col].fill_between(range(ref_img.shape[1]),
                               ref_img[row_m,:].astype(float),
                               img_saved[row_m,:].astype(float),
                               alpha=0.2, color='red')
    axes[3, col].set_title('Profil Baris Tengah', fontsize=8)
    axes[3, col].set_xlabel('Kolom', fontsize=8)
    axes[3, col].set_ylabel('Nilai Piksel', fontsize=8)
    axes[3, col].set_xlim(0, ref_img.shape[1])
    axes[3, col].set_ylim(-5, 265)
    axes[3, col].legend(fontsize=7)
    axes[3, col].grid(True, alpha=0.3)

row_labels_fmt = ['Citra Hasil', 'Error Map', 'Histogram', 'Profil Baris']
for row, label in enumerate(row_labels_fmt):
    fig.text(0.005, 0.87 - row*0.215, label, fontsize=10,
             fontweight='bold', rotation=90, va='center', color='#1A237E')

plt.suptitle('Perbandingan Format Kompresi: PNG (Lossless) vs JPEG (Lossy)\n'
             '(Hijau = lossless | Merah = lossy)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0.025, 0, 1, 0.97])
plt.savefig('cell15_format_strips.png', dpi=100, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Visualisasi 2: Bar chart perbandingan format
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fmt_labels = list(format_results.keys())
cr_fmt   = [format_results[f]['compression_ratio'] for f in fmt_labels]
psnr_fmt = [format_results[f]['psnr'] if format_results[f]['psnr']!=float('inf') else 52
            for f in fmt_labels]
ssim_fmt = [format_results[f]['ssim'] for f in fmt_labels]
type_col = ['#1565C0' if format_results[f]['type']=='lossless' else '#B71C1C'
            for f in fmt_labels]

bars1 = axes[0].bar(fmt_labels, cr_fmt, color=type_col, edgecolor='black', linewidth=0.8)
axes[0].set_title('Compression Ratio\n(Biru=Lossless | Merah=Lossy)', fontweight='bold')
axes[0].set_ylabel('CR')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, cr_fmt):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}x', ha='center', fontsize=9, fontweight='bold')

bars2 = axes[1].bar(fmt_labels, psnr_fmt, color=type_col, edgecolor='black', linewidth=0.8)
axes[1].axhline(y=30, color='orange', linestyle='--', linewidth=1.5)
axes[1].axhline(y=40, color='green',  linestyle='--', linewidth=1.5)
axes[1].axhline(y=52, color='gray',   linestyle=':',  linewidth=1.5, label='52=inf/lossless')
axes[1].set_title('PSNR (dB)\n(52 = infiniti / lossless)', fontweight='bold')
axes[1].set_ylabel('PSNR (dB)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, psnr_fmt):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')

bars3 = axes[2].bar(fmt_labels, ssim_fmt, color=type_col, edgecolor='black', linewidth=0.8)
axes[2].axhline(y=0.9,  color='orange', linestyle='--', linewidth=1.5, label='SSIM=0.9')
axes[2].axhline(y=0.95, color='green',  linestyle='--', linewidth=1.5, label='SSIM=0.95')
axes[2].set_title('SSIM\n(1.0 = identik)', fontweight='bold')
axes[2].set_ylabel('SSIM')
axes[2].set_ylim(0, 1.1)
axes[2].tick_params(axis='x', rotation=20)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, ssim_fmt):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Ringkasan Metrik: PNG vs JPEG pada Berbagai Quality Factor',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cell15_format_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

# Cleanup
for fmt_name, data in format_results.items():
    if os.path.exists(data['path']):
        os.remove(data['path'])
print("File sementara dihapus.")

In [ ]:
# Cell 16: Ringkasan akhir + visualisasi infografis

def print_final_summary(comp_results, test_images):
    print("=" * 70)
    print("         RINGKASAN AKHIR PRAKTIKUM KOMPRESI CITRA DIGITAL")
    print("=" * 70)
    print("\n1. EFEKTIVITAS RLE (Lossless):")
    for img_name in test_images.keys():
        enc, _, _ = rle_encode_image(test_images[img_name])
        rle_b = calculate_rle_size(enc)
        orig_b = test_images[img_name].nbytes
        cr = calculate_compression_ratio(orig_b, rle_b)
        flag = "(efektif)" if cr > 1 else "(TIDAK efektif)"
        print(f"   {img_name:<20}: CR = {cr:.2f} {flag}")

    print("\n2. PANDUAN PEMILIHAN METODE:")
    print("   Citra Medis    -> Lossless (PNG)       [tidak ada kehilangan data]")
    print("   Foto Digital   -> JPEG QF=80-90        [keseimbangan kualitas-ukuran]")
    print("   Thumbnail Web  -> JPEG QF=50-70        [prioritas ukuran kecil]")
    print("   Dokumen Teks   -> PNG / RLE             [lossless untuk keterbacaan]")
    print("   Citra Ilmiah   -> Lossless (PNG/TIFF)  [presisi data penting]")
    print("=" * 70)

print_final_summary(comp_results, test_images)

# -------------------------------------------------------
# Visualisasi: Infografis rangkuman metode
# -------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.patch.set_facecolor('#F5F5F5')

# ---- Subplot 1: Perbandingan paradigma Lossless vs Lossy ----
ax1 = axes[0, 0]
ax1.axis('off')
ax1.set_facecolor('#FAFAFA')

sections = [
    (0.05, 0.50, 0.43, 0.95, '#E3F2FD', '#1565C0', 'LOSSLESS',
     ['RLE', 'Zlib / PNG', 'TIFF',
      '', 'PSNR = inf', 'SSIM = 1.0',
      'CR: 1x - 5x',
      '', 'Cocok untuk:', 'Dokumen, Medis,', 'Citra Ilmiah']),
    (0.52, 0.50, 0.43, 0.95, '#FBE9E7', '#B71C1C', 'LOSSY',
     ['DCT / JPEG', 'WebP', 'HEIC',
      '', 'PSNR: 20-45 dB', 'SSIM: 0.7-0.99',
      'CR: 5x - 50x',
      '', 'Cocok untuk:', 'Foto, Video,', 'Multimedia Web']),
]

for x, y, w, h_r, bg, tc, title, items in sections:
    ax1.add_patch(mpatches.FancyBboxPatch(
        (x, y), w, h_r, transform=ax1.transAxes,
        boxstyle='round,pad=0.02', facecolor=bg,
        edgecolor=tc, linewidth=2.5, clip_on=False
    ))
    ax1.text(x+w/2, y+h_r-0.05, title, transform=ax1.transAxes,
             ha='center', fontsize=16, fontweight='bold', color=tc)
    for i, item in enumerate(items):
        ax1.text(x+w/2, y+h_r-0.14 - i*0.06, item, transform=ax1.transAxes,
                 ha='center', fontsize=9.5,
                 color=tc if item else 'gray',
                 fontweight='bold' if item.startswith('Cocok') else 'normal')

ax1.text(0.5, 0.42, 'VS', transform=ax1.transAxes, ha='center',
         fontsize=22, fontweight='bold', color='#424242')

ax1.text(0.5, 0.20, 'TRADE-OFF UTAMA', transform=ax1.transAxes,
         ha='center', fontsize=13, fontweight='bold', color='#424242')
ax1.text(0.5, 0.10,
         'Kualitas Tinggi  <---------->  Kompresi Tinggi',
         transform=ax1.transAxes, ha='center', fontsize=11, color='#555555')
ax1.text(0.5, 0.03,
         'Pilih sesuai kebutuhan aplikasi',
         transform=ax1.transAxes, ha='center', fontsize=10,
         color='#888888', style='italic')

# ---- Subplot 2: Timeline proses JPEG ----
ax2 = axes[0, 1]
ax2.axis('off')
ax2.set_facecolor('#FFF8E1')

jpeg_steps = [
    (0.08, 'Citra Asli\n(Piksel)',         '#1565C0', 'Input'),
    (0.22, 'Bagi\nBlok 8x8',               '#1976D2', 'Pre-process'),
    (0.36, 'DCT\n(Spasial->Frek)',         '#0288D1', 'Transform'),
    (0.50, 'Kuantisasi\n(LOSSY)',           '#E64A19', 'Quantize'),
    (0.64, 'Entropy\nCoding',              '#388E3C', 'Encode'),
    (0.78, 'File\nTerkompresi',            '#1B5E20', 'Output'),
]

for i, (x, label, color, tag) in enumerate(jpeg_steps):
    circle = plt.Circle((x, 0.62), 0.065, transform=ax2.transAxes,
                         color=color, zorder=5)
    ax2.add_artist(circle)
    ax2.text(x, 0.62, str(i+1), transform=ax2.transAxes,
             ha='center', va='center', fontsize=14,
             fontweight='bold', color='white', zorder=6)
    ax2.text(x, 0.46, label, transform=ax2.transAxes,
             ha='center', va='top', fontsize=8.5,
             fontweight='bold', color=color)
    ax2.text(x, 0.90, tag, transform=ax2.transAxes,
             ha='center', va='center', fontsize=8,
             color='gray', style='italic')

    if i < len(jpeg_steps)-1:
        ax2.annotate('', xy=(jpeg_steps[i+1][0]-0.065, 0.62),
                      xytext=(x+0.065, 0.62),
                      xycoords='axes fraction', textcoords='axes fraction',
                      arrowprops=dict(arrowstyle='->', color='#424242', lw=2))

ax2.text(0.5, 0.26,
         'Kuantisasi adalah satu-satunya tahap yang bersifat LOSSY\n'
         'Quality Factor menentukan seberapa agresif kuantisasi dilakukan',
         transform=ax2.transAxes, ha='center', fontsize=10,
         color='#B71C1C', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#FFECB3', alpha=0.8))

ax2.text(0.5, 0.08,
         'Proses Dekompres: Dequantisasi -> IDCT -> Gabung Blok -> Citra Rekonstruksi',
         transform=ax2.transAxes, ha='center', fontsize=9, color='#555555')

ax2.set_title('Alur Proses Kompresi DCT/JPEG',
              fontweight='bold', fontsize=12, pad=10)

# ---- Subplot 3: Panduan interpreasi PSNR + SSIM ----
ax3 = axes[1, 0]
ax3.axis('off')

psnr_guide = [
    ('> 40 dB',  '#2E7D32', 'Sangat Baik', 'Hampir tidak terlihat perbedaannya'),
    ('30-40 dB', '#558B2F', 'Baik',         'Perbedaan minimal, dapat diterima'),
    ('20-30 dB', '#F57F17', 'Cukup',        'Perbedaan terlihat namun masih berguna'),
    ('< 20 dB',  '#B71C1C', 'Buruk',        'Kualitas rendah, artefak jelas terlihat'),
]

ax3.text(0.5, 0.97, 'Panduan Interpretasi Kualitas Kompresi',
         transform=ax3.transAxes, ha='center', fontsize=13,
         fontweight='bold', color='#212121')

for i, (psnr_range, color, quality, desc) in enumerate(psnr_guide):
    y_pos = 0.82 - i * 0.18
    ax3.add_patch(mpatches.FancyBboxPatch(
        (0.03, y_pos-0.06), 0.94, 0.14,
        transform=ax3.transAxes, boxstyle='round,pad=0.02',
        facecolor=color, alpha=0.15, edgecolor=color, linewidth=1.5,
        clip_on=False
    ))
    ax3.text(0.07, y_pos+0.01, psnr_range, transform=ax3.transAxes,
             fontsize=12, fontweight='bold', color=color, va='center')
    ax3.text(0.28, y_pos+0.01, quality, transform=ax3.transAxes,
             fontsize=11, fontweight='bold', color=color, va='center')
    ax3.text(0.50, y_pos+0.01, desc, transform=ax3.transAxes,
             fontsize=10, color='#424242', va='center')

ax3.add_patch(mpatches.FancyBboxPatch(
    (0.03, 0.07), 0.94, 0.15,
    transform=ax3.transAxes, boxstyle='round,pad=0.02',
    facecolor='#E8EAF6', edgecolor='#3949AB', linewidth=1.5, clip_on=False
))
ax3.text(0.5, 0.16, 'SSIM mendekati 1.0 = struktur gambar sangat mirip aslinya',
         transform=ax3.transAxes, ha='center', fontsize=10,
         color='#3949AB', fontweight='bold')
ax3.text(0.5, 0.09, 'Kombinasi PSNR tinggi + SSIM tinggi = kompresi berkualitas',
         transform=ax3.transAxes, ha='center', fontsize=9.5, color='#555555')

ax3.set_title('Interpretasi Metrik Evaluasi', fontweight='bold', fontsize=12, pad=10)

# ---- Subplot 4: Ringkasan metrik semua metode pada satu citra ----
ax4 = axes[1, 1]
ref_name  = 'geometric'
methods_s = list(method_info.keys())
cr_s_plot = [comp_results[ref_name][m]['compression_ratio'] for m in methods_s]
psnr_s_plot = [comp_results[ref_name][m]['psnr'] if comp_results[ref_name][m]['psnr'] != float('inf') else 52
               for m in methods_s]
ssim_s_plot = [comp_results[ref_name][m]['ssim'] for m in methods_s]
colors_s    = [method_info[m]['color'] for m in methods_s]

ax4_twin = ax4.twinx()
w_s = 0.25
x_s = np.arange(len(methods_s))

bars_cr   = ax4.bar(x_s - w_s, cr_s_plot, w_s, color=colors_s,
                     alpha=0.9, edgecolor='black', linewidth=0.7, label='CR (kiri)')
bars_psnr = ax4_twin.bar(x_s, psnr_s_plot, w_s, color=colors_s,
                          alpha=0.5, edgecolor='black', linewidth=0.7, label='PSNR (kanan)')
bars_ssim = ax4_twin.bar(x_s + w_s, [s*50 for s in ssim_s_plot], w_s,
                          color=colors_s, alpha=0.3, edgecolor='black',
                          linewidth=0.7, hatch='//', label='SSIM*50 (kanan)')

ax4.set_xticks(x_s)
ax4.set_xticklabels(methods_s, rotation=30, ha='right', fontsize=9)
ax4.set_ylabel('Compression Ratio', color='black', fontsize=10)
ax4_twin.set_ylabel('PSNR (dB) / SSIM*50', color='gray', fontsize=10)
ax4.set_title(f'Ringkasan Semua Metrik\nCitra: {ref_name}',
              fontweight='bold', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')

lossless_patch = mpatches.Patch(color='#1565C0', label='Lossless Methods')
lossy_patch    = mpatches.Patch(color='#B71C1C', label='Lossy Methods')
cr_line   = mpatches.Patch(color='gray', alpha=0.9, label='CR (bar gelap)')
psnr_line = mpatches.Patch(color='gray', alpha=0.5, label='PSNR (bar sedang)')
ssim_line = mpatches.Patch(color='gray', alpha=0.3, hatch='//', label='SSIM*50 (bar terang)')
ax4.legend(handles=[lossless_patch, lossy_patch, cr_line, psnr_line, ssim_line],
           fontsize=8, loc='upper right', ncol=2)

plt.suptitle('Infografis Rangkuman: Praktikum Kompresi Citra Digital',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cell16_summary_infographic.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nPraktikum selesai. Semua visualisasi telah disimpan.")
print("Pastikan semua cell dijalankan sebelum menyerahkan laporan.")